<a href="https://colab.research.google.com/github/Ashraf310120/Python_Assignments_DS_/blob/main/Edukron_Python_12_Exception_Handling(51_100).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## === Q51. Create insufficientfundserror for banking case 3; use sample reference ACC-1051 and explain the result. ===

In [ ]:
class InsufficientFundsError(Exception):
    """Raised when an account has insufficient funds."""
    pass


account_number = "ACC-1051"
balance = 8000
withdrawal_amount = 10000

try:
    # Check available balance
    if withdrawal_amount > balance:
        raise InsufficientFundsError(
            "Insufficient funds for this withdrawal."
        )

    # Update balance only after validation succeeds
    balance -= withdrawal_amount

    print("Withdrawal successful.")
    print("Account Number:", account_number)
    print("Withdrawal Amount:", withdrawal_amount)
    print("Remaining Balance:", balance)

except InsufficientFundsError as error:
    print("Withdrawal rejected:", error)

Withdrawal rejected: Insufficient funds for this withdrawal.


## === Q52. Chain invalidtransactionerror from valueerror for banking case 3; use sample reference ACC-1052 and explain the result. ===

In [ ]:
class InvalidTransactionError(Exception):
    """Raised when a banking transaction is invalid."""
    pass


account_number = "ACC-1052"
balance = 10000
amount_input = "abc"

try:
    try:
        # Risky statement: invalid input can raise ValueError
        amount = float(amount_input)

        # Business-rule validation
        if amount <= 0:
            raise ValueError(
                "Transaction amount must be greater than zero."
            )

    except ValueError as error:
        # Chain the original ValueError
        raise InvalidTransactionError(
            "Invalid transaction amount."
        ) from error

    # Update balance only after validation succeeds
    balance -= amount

    print("Transaction successful.")
    print("Account Number:", account_number)
    print("Transaction Amount:", amount)
    print("Remaining Balance:", balance)

except InvalidTransactionError as error:
    print("Transaction rejected:", error)
    print("Original cause:", error.__cause__)

Transaction rejected: Invalid transaction amount.
Original cause: could not convert string to float: 'abc'


## === Q53. Process transfers while isolating bad rows for banking case 3; use sample reference ACC-1053 and explain the result. ===

In [3]:
class InvalidTransactionError(Exception):
    """Raised when a transfer is invalid."""
    pass


account_number = "ACC-1053"
balance = 10000

transfers = [
    {"reference": "TXN-001", "amount": 2000},
    {"reference": "TXN-002", "amount": "abc"},
    {"reference": "TXN-003", "amount": 3000},
    {"reference": "TXN-004", "amount": -500},
    {"reference": "TXN-005", "amount": 4000}
]

successful_transfers = []
failed_transfers = []

for transfer in transfers:
    try:
        # Risky statement: invalid values can raise ValueError
        amount = float(transfer["amount"])

        # Business-rule validation
        if amount <= 0:
            raise InvalidTransactionError(
                "Transfer amount must be greater than zero."
            )

        if amount > balance:
            raise InvalidTransactionError(
                "Insufficient funds for this transfer."
            )

        # Update balance only after validation succeeds
        balance -= amount

        successful_transfers.append(transfer["reference"])

    except (ValueError, InvalidTransactionError) as error:
        # Record the bad row and continue processing
        failed_transfers.append(
            (transfer["reference"], str(error))
        )


print("Account Number:", account_number)
print("Final Balance:", balance)

print("\nSuccessful Transfers:")
for reference in successful_transfers:
    print(reference)

print("\nFailed Transfers:")
for reference, error in failed_transfers:
    print(reference, "-", error)

Account Number: ACC-1053
Final Balance: 1000.0

Successful Transfers:
TXN-001
TXN-003
TXN-005

Failed Transfers:
TXN-002 - could not convert string to float: 'abc'
TXN-004 - Transfer amount must be greater than zero.


## === Q54. Retry a transient payment gateway for banking case 3; use sample reference ACC-1054 and explain the result. ===

In [4]:
class TransientGatewayError(Exception):
    """Raised when the payment gateway temporarily fails."""
    pass


def payment_gateway():
    # Simulate a temporary gateway failure
    payment_gateway.attempts += 1

    if payment_gateway.attempts < 3:
        raise TransientGatewayError(
            "Payment gateway temporarily unavailable."
        )

    return True


payment_gateway.attempts = 0

account_number = "ACC-1054"
balance = 10000
payment_amount = 3000
max_retries = 3

for attempt in range(1, max_retries + 1):
    try:
        # Try the external payment gateway
        payment_gateway()

        # Update balance only after successful payment
        balance -= payment_amount

        print("Payment successful.")
        print("Account Number:", account_number)
        print("Payment Amount:", payment_amount)
        print("Remaining Balance:", balance)

        break

    except TransientGatewayError as error:
        print(f"Attempt {attempt} failed:", error)

        if attempt == max_retries:
            print("Payment failed after all retry attempts.")

Attempt 1 failed: Payment gateway temporarily unavailable.
Attempt 2 failed: Payment gateway temporarily unavailable.
Payment successful.
Account Number: ACC-1054
Payment Amount: 3000
Remaining Balance: 7000


## === Q55. Log failed standing instructions for banking case 3; use sample reference ACC-1055 and explain the result. ===

In [5]:
import logging


class StandingInstructionError(Exception):
    """Raised when a standing instruction fails."""
    pass


# Configure logging
logging.basicConfig(
    filename="standing_instructions.log",
    level=logging.ERROR,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


account_number = "ACC-1055"
balance = 5000
instruction_amount = 7000


try:
    # Validate the standing instruction amount
    if not isinstance(instruction_amount, (int, float)):
        raise StandingInstructionError(
            "Instruction amount must be a number."
        )

    if instruction_amount <= 0:
        raise StandingInstructionError(
            "Instruction amount must be greater than zero."
        )

    # Check balance before making any deduction
    if instruction_amount > balance:
        raise StandingInstructionError(
            "Insufficient funds for standing instruction."
        )

    # Update balance only after all validations pass
    balance -= instruction_amount

    print("Standing instruction successful.")
    print("Account Number:", account_number)
    print("Instruction Amount:", instruction_amount)
    print("Remaining Balance:", balance)


except StandingInstructionError as error:
    # Log the business-rule failure
    logging.error(
        "Account %s: %s",
        account_number,
        error
    )

    print("Standing instruction failed:", error)
    print("Failure logged successfully.")
    print("Account Balance:", balance)

ERROR:root:Account ACC-1055: Insufficient funds for standing instruction.


Standing instruction failed: Insufficient funds for standing instruction.
Failure logged successfully.
Account Balance: 5000


## ===Q56. Return a structured failure result for banking case 3; use sample reference ACC-1056 and explain the result. ===

In [6]:
class InvalidTransactionError(Exception):
    """Raised when a banking transaction is invalid."""
    pass


def process_transaction(account_number, balance, amount):
    try:
        # Validate transaction amount
        if not isinstance(amount, (int, float)):
            raise InvalidTransactionError(
                "Transaction amount must be a number."
            )

        if amount <= 0:
            raise InvalidTransactionError(
                "Transaction amount must be greater than zero."
            )

        # Check available balance
        if amount > balance:
            raise InvalidTransactionError(
                "Insufficient funds."
            )

        # Update balance only after validation succeeds
        balance -= amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "amount": amount,
            "remaining_balance": balance
        }

    except InvalidTransactionError as error:
        return {
            "status": "FAILED",
            "account_number": account_number,
            "amount": amount,
            "error": str(error),
            "remaining_balance": balance
        }


account_number = "ACC-1056"
balance = 5000
withdrawal_amount = 7000

result = process_transaction(
    account_number,
    balance,
    withdrawal_amount
)

print(result)

{'status': 'FAILED', 'account_number': 'ACC-1056', 'amount': 7000, 'error': 'Insufficient funds.', 'remaining_balance': 5000}


## === Q57. Validate a nested beneficiary record for banking case 3; use sample reference ACC-1057 and explain the result. ===

In [7]:
class InvalidBeneficiaryError(Exception):
    """Raised when beneficiary details are invalid."""
    pass


account_number = "ACC-1057"

beneficiary = {
    "details": {
        "name": "Rahul",
        "account_number": "BEN-1001"
    }
}


try:
    # Access the nested beneficiary details
    beneficiary_name = beneficiary["details"]["name"]
    beneficiary_account = beneficiary["details"]["account_number"]

    # Validate beneficiary name
    if not isinstance(beneficiary_name, str):
        raise InvalidBeneficiaryError(
            "Beneficiary name must be text."
        )

    if not beneficiary_name.strip():
        raise InvalidBeneficiaryError(
            "Beneficiary name cannot be empty."
        )

    # Validate beneficiary account number
    if not isinstance(beneficiary_account, str):
        raise InvalidBeneficiaryError(
            "Beneficiary account number must be text."
        )

    if not beneficiary_account.strip():
        raise InvalidBeneficiaryError(
            "Beneficiary account number cannot be empty."
        )

    print("Beneficiary validation successful.")
    print("Account Number:", account_number)
    print("Beneficiary Name:", beneficiary_name)
    print("Beneficiary Account:", beneficiary_account)


except KeyError as error:
    print("Beneficiary record is missing a required field:", error)

except InvalidBeneficiaryError as error:
    print("Beneficiary validation failed:", error)

Beneficiary validation successful.
Account Number: ACC-1057
Beneficiary Name: Rahul
Beneficiary Account: BEN-1001


## === Q58. Use a context manager for an audit file for banking case 3; use sample reference ACC-1058 and explain the result. ===

In [8]:
account_number = "ACC-1058"
file_name = "audit_log.txt"

try:
    # Context manager automatically closes the file
    with open(file_name, "a") as audit_file:

        audit_entry = (
            f"Audit entry: Transaction checked for {account_number}\n"
        )

        audit_file.write(audit_entry)

        print("Audit entry written successfully.")
        print("Account Number:", account_number)

except OSError as error:
    print("Unable to access audit file:", error)

Audit entry written successfully.
Account Number: ACC-1058


## === Q59. Separate validation and recovery for banking case 3; use sample reference ACC-1059 and explain the result. ===

In [9]:
class InvalidTransactionError(Exception):
    """Raised when a transaction fails validation."""
    pass


def validate_transaction(amount):
    """Validate the transaction amount."""
    if not isinstance(amount, (int, float)):
        raise InvalidTransactionError(
            "Transaction amount must be a number."
        )

    if amount <= 0:
        raise InvalidTransactionError(
            "Transaction amount must be greater than zero."
        )

    return True


def process_transaction(account_number, balance, amount):
    """Validate and process a withdrawal."""
    try:
        # Validation is handled separately
        validate_transaction(amount)

        # Check balance before changing it
        if amount > balance:
            raise InvalidTransactionError(
                "Insufficient funds."
            )

        # Update balance only after validation succeeds
        balance -= amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "amount": amount,
            "remaining_balance": balance
        }

    except InvalidTransactionError as error:
        # Recovery/handling is kept separate from validation
        return {
            "status": "FAILED",
            "account_number": account_number,
            "amount": amount,
            "error": str(error),
            "remaining_balance": balance
        }


account_number = "ACC-1059"
balance = 5000
withdrawal_amount = 7000

result = process_transaction(
    account_number,
    balance,
    withdrawal_amount
)

print(result)

{'status': 'FAILED', 'account_number': 'ACC-1059', 'amount': 7000, 'error': 'Insufficient funds.', 'remaining_balance': 5000}


## === Q60. Collect all kyc validation errors for banking case 3; use sample reference ACC-1060 and explain the result. ===

In [10]:
class KYCValidationError(Exception):
    """Raised when KYC validation fails."""
    pass


def validate_kyc(customer):
    """Validate all required KYC fields and collect errors."""
    errors = []

    # Validate name
    name = customer.get("name", "")

    if not isinstance(name, str):
        errors.append("Name must be text.")
    elif not name.strip():
        errors.append("Name is required.")

    # Validate age
    age = customer.get("age")

    if age is None:
        errors.append("Age is required.")
    elif not isinstance(age, int):
        errors.append("Age must be an integer.")
    elif age < 18:
        errors.append("Customer must be at least 18 years old.")

    # Validate address
    address = customer.get("address", "")

    if not isinstance(address, str):
        errors.append("Address must be text.")
    elif not address.strip():
        errors.append("Address is required.")

    # Raise one exception containing all validation errors
    if errors:
        raise KYCValidationError(errors)

    return True


account_number = "ACC-1060"

customer = {
    "name": "",
    "age": 16,
    "address": ""
}


try:
    validate_kyc(customer)

    print("KYC validation successful.")
    print("Account Number:", account_number)

except KYCValidationError as error:
    print("KYC validation failed.")

    for message in error.args[0]:
        print("-", message)

KYC validation failed.
- Name is required.
- Customer must be at least 18 years old.
- Address is required.


## === Q61. Create insufficientfundserror for banking case 4; use sample reference ACC-1061 and explain the result. ===

In [11]:
class InsufficientFundsError(Exception):
    """Raised when an account has insufficient funds."""
    pass


account_number = "ACC-1061"
balance = 8000
withdrawal_amount = 10000


try:
    # Check whether sufficient balance is available
    if withdrawal_amount > balance:
        raise InsufficientFundsError(
            "Insufficient funds for this withdrawal."
        )

    # Update balance only after the check succeeds
    balance -= withdrawal_amount

    print("Withdrawal successful.")
    print("Account Number:", account_number)
    print("Withdrawal Amount:", withdrawal_amount)
    print("Remaining Balance:", balance)


except InsufficientFundsError as error:
    print("Withdrawal rejected:", error)
    print("Account Balance:", balance)

Withdrawal rejected: Insufficient funds for this withdrawal.
Account Balance: 8000


## === Q62. Chain invalidtransactionerror from valueerror for banking case 4; use sample reference ACC-1062 and explain the result. ===

In [12]:
class InvalidTransactionError(Exception):
    """Raised when a banking transaction is invalid."""
    pass


account_number = "ACC-1062"
balance = 10000
amount_input = "abc"


try:
    try:
        # Convert the input into a numeric amount
        amount = float(amount_input)

        # Validate the transaction amount
        if amount <= 0:
            raise ValueError(
                "Transaction amount must be greater than zero."
            )

    except ValueError as error:
        # Chain the original ValueError
        raise InvalidTransactionError(
            "Invalid transaction amount."
        ) from error

    # Update balance only after successful validation
    balance -= amount

    print("Transaction successful.")
    print("Account Number:", account_number)
    print("Transaction Amount:", amount)
    print("Remaining Balance:", balance)


except InvalidTransactionError as error:
    print("Transaction rejected:", error)
    print("Original cause:", error.__cause__)
    print("Account Balance:", balance)

Transaction rejected: Invalid transaction amount.
Original cause: could not convert string to float: 'abc'
Account Balance: 10000


## === Q63. Process transfers while isolating bad rows for banking case 4; use sample reference ACC-1063 and explain the result. ===

In [13]:
class InvalidTransactionError(Exception):
    """Raised when a transfer is invalid."""
    pass


account_number = "ACC-1063"
balance = 10000

transfers = [
    {"reference": "TXN-001", "amount": 2000},
    {"reference": "TXN-002", "amount": "abc"},
    {"reference": "TXN-003", "amount": 3000},
    {"reference": "TXN-004", "amount": -500},
    {"reference": "TXN-005", "amount": 4000}
]

successful_transfers = []
failed_transfers = []


for transfer in transfers:
    try:
        # Convert the transfer amount to a number
        amount = float(transfer["amount"])

        # Validate the transfer amount
        if amount <= 0:
            raise InvalidTransactionError(
                "Transfer amount must be greater than zero."
            )

        # Check available balance
        if amount > balance:
            raise InvalidTransactionError(
                "Insufficient funds for this transfer."
            )

        # Update balance only after all checks pass
        balance -= amount

        successful_transfers.append(
            transfer["reference"]
        )

    except (ValueError, InvalidTransactionError) as error:
        # Isolate the bad row and continue processing
        failed_transfers.append(
            (transfer["reference"], str(error))
        )


print("Account Number:", account_number)
print("Final Balance:", balance)

print("\nSuccessful Transfers:")
for reference in successful_transfers:
    print(reference)

print("\nFailed Transfers:")
for reference, error in failed_transfers:
    print(reference, "-", error)

Account Number: ACC-1063
Final Balance: 1000.0

Successful Transfers:
TXN-001
TXN-003
TXN-005

Failed Transfers:
TXN-002 - could not convert string to float: 'abc'
TXN-004 - Transfer amount must be greater than zero.


## === Q64. Retry a transient payment gateway for banking case 4; use sample reference ACC-1064 and explain the result. ===

In [14]:
class TransientGatewayError(Exception):
    """Raised when the payment gateway temporarily fails."""
    pass


def payment_gateway():
    """Simulate a temporary payment gateway failure."""
    payment_gateway.attempts += 1

    # Gateway fails for the first two attempts
    if payment_gateway.attempts < 3:
        raise TransientGatewayError(
            "Payment gateway temporarily unavailable."
        )

    return True


payment_gateway.attempts = 0

account_number = "ACC-1064"
balance = 10000
payment_amount = 3000
max_retries = 3

for attempt in range(1, max_retries + 1):
    try:
        # Try the external payment gateway
        payment_gateway()

        # Deduct money only after successful payment
        balance -= payment_amount

        print("Payment successful.")
        print("Account Number:", account_number)
        print("Payment Amount:", payment_amount)
        print("Remaining Balance:", balance)

        break

    except TransientGatewayError as error:
        print(f"Attempt {attempt} failed:", error)

        if attempt == max_retries:
            print("Payment failed after all retry attempts.")

Attempt 1 failed: Payment gateway temporarily unavailable.
Attempt 2 failed: Payment gateway temporarily unavailable.
Payment successful.
Account Number: ACC-1064
Payment Amount: 3000
Remaining Balance: 7000


## === Q65. Log failed standing instructions for banking case 4; use sample reference ACC-1065 and explain the result. ===

In [15]:
import logging


class StandingInstructionError(Exception):
    """Raised when a standing instruction fails."""
    pass


# Configure logging
logging.basicConfig(
    filename="standing_instructions.log",
    level=logging.ERROR,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


account_number = "ACC-1065"
balance = 5000
instruction_amount = 7000

try:
    # Validate instruction amount
    if not isinstance(instruction_amount, (int, float)):
        raise StandingInstructionError(
            "Instruction amount must be a number."
        )

    if instruction_amount <= 0:
        raise StandingInstructionError(
            "Instruction amount must be greater than zero."
        )

    # Check available balance
    if instruction_amount > balance:
        raise StandingInstructionError(
            "Insufficient funds for standing instruction."
        )

    # Update balance only after all validations pass
    balance -= instruction_amount

    print("Standing instruction successful.")
    print("Account Number:", account_number)
    print("Instruction Amount:", instruction_amount)
    print("Remaining Balance:", balance)

except StandingInstructionError as error:
    # Log the failure
    logging.error(
        "Account %s: %s",
        account_number,
        error
    )

    print("Standing instruction failed:", error)
    print("Failure logged successfully.")
    print("Account Balance:", balance)

ERROR:root:Account ACC-1065: Insufficient funds for standing instruction.


Standing instruction failed: Insufficient funds for standing instruction.
Failure logged successfully.
Account Balance: 5000


## === Q66. Return a structured failure result for banking case 4; use sample reference ACC-1066 and explain the result. ===

In [16]:
class InvalidTransactionError(Exception):
    """Raised when a banking transaction is invalid."""
    pass


def process_transaction(account_number, balance, amount):
    try:
        # Validate transaction amount
        if not isinstance(amount, (int, float)):
            raise InvalidTransactionError(
                "Transaction amount must be a number."
            )

        if amount <= 0:
            raise InvalidTransactionError(
                "Transaction amount must be greater than zero."
            )

        # Check available balance
        if amount > balance:
            raise InvalidTransactionError(
                "Insufficient funds."
            )

        # Update balance only after validation succeeds
        balance -= amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "amount": amount,
            "remaining_balance": balance
        }

    except InvalidTransactionError as error:
        return {
            "status": "FAILED",
            "account_number": account_number,
            "amount": amount,
            "error": str(error),
            "remaining_balance": balance
        }


account_number = "ACC-1066"
balance = 5000
withdrawal_amount = 7000

result = process_transaction(
    account_number,
    balance,
    withdrawal_amount
)

print(result)

{'status': 'FAILED', 'account_number': 'ACC-1066', 'amount': 7000, 'error': 'Insufficient funds.', 'remaining_balance': 5000}


## === Q67. Validate a nested beneficiary record for banking case 4; use sample reference ACC-1067 and explain the result. ===

In [17]:
class InvalidBeneficiaryError(Exception):
    """Raised when beneficiary details are invalid."""
    pass


account_number = "ACC-1067"

beneficiary = {
    "details": {
        "name": "Rahul",
        "account_number": "BEN-1001"
    }
}

try:
    # Access the nested beneficiary details
    beneficiary_name = beneficiary["details"]["name"]
    beneficiary_account = beneficiary["details"]["account_number"]

    # Validate beneficiary name
    if not isinstance(beneficiary_name, str):
        raise InvalidBeneficiaryError(
            "Beneficiary name must be text."
        )

    if not beneficiary_name.strip():
        raise InvalidBeneficiaryError(
            "Beneficiary name cannot be empty."
        )

    # Validate beneficiary account number
    if not isinstance(beneficiary_account, str):
        raise InvalidBeneficiaryError(
            "Beneficiary account number must be text."
        )

    if not beneficiary_account.strip():
        raise InvalidBeneficiaryError(
            "Beneficiary account number cannot be empty."
        )

    print("Beneficiary validation successful.")
    print("Account Number:", account_number)
    print("Beneficiary Name:", beneficiary_name)
    print("Beneficiary Account:", beneficiary_account)

except KeyError as error:
    print(
        "Beneficiary record is missing a required field:",
        error
    )

except InvalidBeneficiaryError as error:
    print("Beneficiary validation failed:", error)

Beneficiary validation successful.
Account Number: ACC-1067
Beneficiary Name: Rahul
Beneficiary Account: BEN-1001


## === Q68. Use a context manager for an audit file for banking case 4; use sample reference ACC-1068 and explain the result. ===

In [18]:
account_number = "ACC-1068"
file_name = "audit_log.txt"

try:
    # Open the audit file safely using a context manager
    with open(file_name, "a") as audit_file:
        audit_entry = (
            f"Audit entry: Transaction checked for "
            f"{account_number}\n"
        )

        audit_file.write(audit_entry)

        print("Audit entry written successfully.")
        print("Account Number:", account_number)

except OSError as error:
    print("Unable to access audit file:", error)

Audit entry written successfully.
Account Number: ACC-1068


## === Q69. Separate validation and recovery for banking case 4; use sample reference ACC-1069 and explain the result. ===

In [19]:
class InvalidTransactionError(Exception):
    """Raised when a transaction fails validation."""
    pass


def validate_transaction(amount):
    """Validate the transaction amount."""

    if not isinstance(amount, (int, float)):
        raise InvalidTransactionError(
            "Transaction amount must be a number."
        )

    if amount <= 0:
        raise InvalidTransactionError(
            "Transaction amount must be greater than zero."
        )

    return True


def process_transaction(account_number, balance, amount):
    """Validate and process a withdrawal."""

    try:
        # Step 1: Validate the transaction
        validate_transaction(amount)

        # Step 2: Apply business rule
        if amount > balance:
            raise InvalidTransactionError(
                "Insufficient funds."
            )

        # Update balance only after validation succeeds
        balance -= amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "amount": amount,
            "remaining_balance": balance
        }

    except InvalidTransactionError as error:
        # Recovery: return a structured failure
        return {
            "status": "FAILED",
            "account_number": account_number,
            "amount": amount,
            "error": str(error),
            "remaining_balance": balance
        }


account_number = "ACC-1069"
balance = 5000
withdrawal_amount = 7000

result = process_transaction(
    account_number,
    balance,
    withdrawal_amount
)

print(result)

{'status': 'FAILED', 'account_number': 'ACC-1069', 'amount': 7000, 'error': 'Insufficient funds.', 'remaining_balance': 5000}


## === Q70. Collect all kyc validation errors for banking case 4; use sample reference ACC-1070 and explain the result. ===

In [20]:
class KYCValidationError(Exception):
    """Raised when KYC validation fails."""
    pass


def validate_kyc(customer):
    errors = []

    # Validate name
    name = customer.get("name", "")

    if not isinstance(name, str):
        errors.append("Name must be text.")
    elif not name.strip():
        errors.append("Name is required.")

    # Validate age
    age = customer.get("age")

    if age is None:
        errors.append("Age is required.")
    elif not isinstance(age, int):
        errors.append("Age must be an integer.")
    elif age < 18:
        errors.append(
            "Customer must be at least 18 years old."
        )

    # Validate address
    address = customer.get("address", "")

    if not isinstance(address, str):
        errors.append("Address must be text.")
    elif not address.strip():
        errors.append("Address is required.")

    # Raise all validation errors together
    if errors:
        raise KYCValidationError(errors)

    return True


account_number = "ACC-1070"

customer = {
    "name": "",
    "age": 16,
    "address": ""
}

try:
    validate_kyc(customer)

    print("KYC validation successful.")
    print("Account Number:", account_number)

except KYCValidationError as error:
    print("KYC validation failed.")

    for message in error.args[0]:
        print("-", message)

KYC validation failed.
- Name is required.
- Customer must be at least 18 years old.
- Address is required.


## === Q71. Design a banking exception hierarchy for banking case 1; use sample reference ACC-1071 and explain the result. ===

In [21]:
# Base exception for banking-related errors
class BankingError(Exception):
    """Base class for banking errors."""
    pass


# Business-rule exceptions
class TransactionError(BankingError):
    """Base class for transaction-related errors."""
    pass


class InsufficientFundsError(TransactionError):
    """Raised when an account has insufficient funds."""
    pass


class InvalidTransactionError(TransactionError):
    """Raised when a transaction is invalid."""
    pass


# Infrastructure-related exception
class PaymentGatewayError(BankingError):
    """Raised when the payment gateway fails."""
    pass


account_number = "ACC-1071"
balance = 5000
withdrawal_amount = 7000

try:
    # Validate transaction amount
    if not isinstance(withdrawal_amount, (int, float)):
        raise InvalidTransactionError(
            "Withdrawal amount must be a number."
        )

    if withdrawal_amount <= 0:
        raise InvalidTransactionError(
            "Withdrawal amount must be greater than zero."
        )

    # Check business rule
    if withdrawal_amount > balance:
        raise InsufficientFundsError(
            "Insufficient funds for this withdrawal."
        )

    # Update balance only after validation succeeds
    balance -= withdrawal_amount

    print("Withdrawal successful.")
    print("Account Number:", account_number)
    print("Withdrawal Amount:", withdrawal_amount)
    print("Remaining Balance:", balance)

except InsufficientFundsError as error:
    print("Withdrawal rejected:", error)
    print("Account Balance:", balance)

except InvalidTransactionError as error:
    print("Invalid transaction:", error)

Withdrawal rejected: Insufficient funds for this withdrawal.
Account Balance: 5000


## === Q72. Make a transfer rollback-safe for banking case 1; use sample reference ACC-1072 and explain the result. ===

In [22]:
class TransferError(Exception):
    """Raised when a bank transfer fails."""
    pass


account_number = "ACC-1072"
beneficiary_account = "BEN-1001"

source_balance = 10000
beneficiary_balance = 3000
transfer_amount = 4000

# Save original balances for rollback
original_source_balance = source_balance
original_beneficiary_balance = beneficiary_balance

try:
    # Validate transfer amount
    if not isinstance(transfer_amount, (int, float)):
        raise TransferError(
            "Transfer amount must be a number."
        )

    if transfer_amount <= 0:
        raise TransferError(
            "Transfer amount must be greater than zero."
        )

    # Check source account balance
    if transfer_amount > source_balance:
        raise TransferError(
            "Insufficient funds in source account."
        )

    # Step 1: Debit source account
    source_balance -= transfer_amount

    # Simulate a failure during the second step
    raise TransferError(
        "Beneficiary account update failed."
    )

    # Step 2: Credit beneficiary account
    beneficiary_balance += transfer_amount

except TransferError as error:
    # Roll back all balance changes
    source_balance = original_source_balance
    beneficiary_balance = original_beneficiary_balance

    print("Transfer failed:", error)
    print("Transfer rolled back successfully.")

print("\nAccount Number:", account_number)
print("Source Balance:", source_balance)
print("Beneficiary Account:", beneficiary_account)
print("Beneficiary Balance:", beneficiary_balance)

Transfer failed: Beneficiary account update failed.
Transfer rolled back successfully.

Account Number: ACC-1072
Source Balance: 10000
Beneficiary Account: BEN-1001
Beneficiary Balance: 3000


## === Q73. Preserve causes across service layers for banking case 1; use sample reference ACC-1073 and explain the result. ===

In [23]:
class PaymentGatewayError(Exception):
    """Raised when the payment gateway fails."""
    pass


class PaymentProcessingError(Exception):
    """Raised when payment processing fails."""
    pass


def payment_gateway(payment_amount):
    """Simulate a payment gateway."""

    if not isinstance(payment_amount, (int, float)):
        raise TypeError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise ValueError(
            "Payment amount must be greater than zero."
        )

    # Simulate a gateway failure
    raise ConnectionError(
        "Payment gateway connection failed."
    )


def process_payment(account_number, balance, payment_amount):
    """Process payment and preserve the original cause."""

    try:
        payment_gateway(payment_amount)

    except (TypeError, ValueError, ConnectionError) as error:
        raise PaymentGatewayError(
            "Payment gateway could not process the payment."
        ) from error


def banking_service(account_number, balance, payment_amount):
    """Higher-level banking service."""

    try:
        process_payment(
            account_number,
            balance,
            payment_amount
        )

        # Balance is updated only after successful payment
        balance -= payment_amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "remaining_balance": balance
        }

    except PaymentGatewayError as error:
        raise PaymentProcessingError(
            "Payment processing failed."
        ) from error


account_number = "ACC-1073"
balance = 10000
payment_amount = 3000

try:
    result = banking_service(
        account_number,
        balance,
        payment_amount
    )

    print(result)

except PaymentProcessingError as error:
    print("Payment failed:", error)
    print("Original cause:", error.__cause__)
    print("Account Balance:", balance)

Payment failed: Payment processing failed.
Original cause: Payment gateway could not process the payment.
Account Balance: 10000


## === Q74. Implement bounded exponential retry for banking case 1; use sample reference ACC-1074 and explain the result. ===

In [24]:
class TransientServiceError(Exception):
    """Raised when a banking service temporarily fails."""
    pass


class PaymentProcessingError(Exception):
    """Raised when payment processing fails after retries."""
    pass


def payment_service():
    """
    Simulate a temporary banking service failure.

    The service fails for the first three attempts
    and succeeds on the fourth attempt.
    """
    payment_service.attempts += 1

    if payment_service.attempts <= 3:
        raise TransientServiceError(
            "Banking service temporarily unavailable."
        )

    return True


payment_service.attempts = 0


def process_payment(
    account_number,
    balance,
    payment_amount,
    max_retries=4,
    base_delay=1,
    max_delay=4
):
    """Process a payment using bounded exponential retry."""

    # Validate business rules before retrying
    if not isinstance(payment_amount, (int, float)):
        raise ValueError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise ValueError(
            "Payment amount must be greater than zero."
        )

    if payment_amount > balance:
        raise ValueError(
            "Insufficient funds."
        )

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            # Smallest risky statement:
            # the external banking service call
            payment_service()

            # Update balance only after service succeeds
            balance -= payment_amount

            return {
                "status": "SUCCESS",
                "account_number": account_number,
                "payment_amount": payment_amount,
                "remaining_balance": balance,
                "attempts": attempt
            }

        except TransientServiceError as error:
            last_error = error

            # Bounded exponential backoff
            delay = min(
                base_delay * (2 ** (attempt - 1)),
                max_delay
            )

            print(
                f"Attempt {attempt} failed. "
                f"Retrying after {delay} second(s)."
            )

            if attempt == max_retries:
                raise PaymentProcessingError(
                    "Payment failed after all retry attempts."
                ) from last_error


# Sample banking case
account_number = "ACC-1074"
balance = 10000
payment_amount = 3000

try:
    result = process_payment(
        account_number,
        balance,
        payment_amount
    )

    print("\nPayment successful.")
    print("Account Number:", result["account_number"])
    print("Payment Amount:", result["payment_amount"])
    print("Remaining Balance:", result["remaining_balance"])
    print("Attempts Used:", result["attempts"])

except PaymentProcessingError as error:
    print("\nPayment failed:", error)
    print("Original cause:", error.__cause__)

except ValueError as error:
    print("\nTransaction rejected:", error)

Attempt 1 failed. Retrying after 1 second(s).
Attempt 2 failed. Retrying after 2 second(s).
Attempt 3 failed. Retrying after 4 second(s).

Payment successful.
Account Number: ACC-1074
Payment Amount: 3000
Remaining Balance: 7000
Attempts Used: 4


## === Q75. Create a transaction context manager for banking case 1; use sample reference ACC-1075 and explain the result. ===

In [25]:
class TransactionError(Exception):
    """Raised when a banking transaction fails."""
    pass


class TransactionContext:
    """Context manager for safe banking transactions."""

    def __init__(self, balance):
        self.original_balance = balance
        self.balance = balance

    def __enter__(self):
        print("Transaction started.")
        return self

    def withdraw(self, amount):
        """Withdraw money after validating the amount."""

        if not isinstance(amount, (int, float)):
            raise TransactionError(
                "Withdrawal amount must be a number."
            )

        if amount <= 0:
            raise TransactionError(
                "Withdrawal amount must be greater than zero."
            )

        if amount > self.balance:
            raise TransactionError(
                "Insufficient funds."
            )

        self.balance -= amount
        print("Withdrawal completed:", amount)

    def deposit(self, amount):
        """Deposit money after validating the amount."""

        if not isinstance(amount, (int, float)):
            raise TransactionError(
                "Deposit amount must be a number."
            )

        if amount <= 0:
            raise TransactionError(
                "Deposit amount must be greater than zero."
            )

        self.balance += amount
        print("Deposit completed:", amount)

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            # No exception occurred, so commit the transaction
            print("Transaction committed.")
            return False

        # An exception occurred, so roll back
        self.balance = self.original_balance

        print("Transaction rolled back.")
        print("Reason:", exc_value)

        # Preserve the original exception
        return False


# Sample banking case
account_number = "ACC-1075"
starting_balance = 10000

try:
    transaction = TransactionContext(starting_balance)

    with transaction:
        transaction.withdraw(3000)
        transaction.deposit(1000)

    print("\nTransaction successful.")
    print("Account Number:", account_number)
    print("Original Balance:", starting_balance)
    print("Final Balance:", transaction.balance)

except TransactionError as error:
    print("\nTransaction failed:", error)

Transaction started.
Withdrawal completed: 3000
Deposit completed: 1000
Transaction committed.

Transaction successful.
Account Number: ACC-1075
Original Balance: 10000
Final Balance: 8000


## === Q76. Build a dead-letter queue for payment events for banking case 1; use sample reference ACC-1076 and explain the result. ===

In [26]:
class TransactionError(Exception):
    """Raised when a banking transaction fails."""
    pass


class TransactionContext:
    """Context manager for safe banking transactions."""

    def __init__(self, balance):
        self.original_balance = balance
        self.balance = balance

    def __enter__(self):
        print("Transaction started.")
        return self

    def withdraw(self, amount):
        """Withdraw money after validating the amount."""

        if not isinstance(amount, (int, float)):
            raise TransactionError(
                "Withdrawal amount must be a number."
            )

        if amount <= 0:
            raise TransactionError(
                "Withdrawal amount must be greater than zero."
            )

        if amount > self.balance:
            raise TransactionError(
                "Insufficient funds."
            )

        self.balance -= amount
        print("Withdrawal completed:", amount)

    def deposit(self, amount):
        """Deposit money after validating the amount."""

        if not isinstance(amount, (int, float)):
            raise TransactionError(
                "Deposit amount must be a number."
            )

        if amount <= 0:
            raise TransactionError(
                "Deposit amount must be greater than zero."
            )

        self.balance += amount
        print("Deposit completed:", amount)

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            # No exception occurred, so commit the transaction
            print("Transaction committed.")
            return False

        # An exception occurred, so roll back
        self.balance = self.original_balance

        print("Transaction rolled back.")
        print("Reason:", exc_value)

        # Preserve the original exception
        return False


# Sample banking case
account_number = "ACC-1075"
starting_balance = 10000

try:
    transaction = TransactionContext(starting_balance)

    with transaction:
        transaction.withdraw(3000)
        transaction.deposit(1000)

    print("\nTransaction successful.")
    print("Account Number:", account_number)
    print("Original Balance:", starting_balance)
    print("Final Balance:", transaction.balance)

except TransactionError as error:
    print("\nTransaction failed:", error)

Transaction started.
Withdrawal completed: 3000
Deposit completed: 1000
Transaction committed.

Transaction successful.
Account Number: ACC-1075
Original Balance: 10000
Final Balance: 8000


## === Q77. Make batch settlement failure-isolated for banking case 1; use sample reference ACC-1077 and explain the result. ===

In [27]:
class SettlementError(Exception):
    """Base class for settlement-related errors."""
    pass


class InvalidSettlementError(SettlementError):
    """Raised when settlement data is invalid."""
    pass


class SettlementServiceError(SettlementError):
    """Raised when the settlement service fails."""
    pass


class SettlementProcessingError(SettlementError):
    """Raised when settlement processing fails."""
    pass


def settlement_service(settlement):
    """
    Simulate an external settlement service.

    TXN-003 fails because of a temporary service problem.
    """

    if settlement["reference"] == "TXN-003":
        raise SettlementServiceError(
            "Settlement service temporarily unavailable."
        )

    return True


def process_settlement_batch(
    account_number,
    balance,
    settlements
):
    """Process each settlement independently."""

    successful_settlements = []
    failed_settlements = []

    for settlement in settlements:
        try:
            # Validate the settlement record
            if not isinstance(settlement, dict):
                raise InvalidSettlementError(
                    "Settlement must be a dictionary."
                )

            reference = settlement.get("reference")
            amount = settlement.get("amount")

            if not reference:
                raise InvalidSettlementError(
                    "Settlement reference is required."
                )

            if not isinstance(amount, (int, float)):
                raise InvalidSettlementError(
                    "Settlement amount must be a number."
                )

            if amount <= 0:
                raise InvalidSettlementError(
                    "Settlement amount must be greater than zero."
                )

            if amount > balance:
                raise InvalidSettlementError(
                    "Insufficient funds."
                )

            # Smallest risky statement
            settlement_service(settlement)

            # Update balance only after successful settlement
            balance -= amount

            successful_settlements.append(reference)

        except SettlementServiceError as error:
            # Preserve the original infrastructure failure
            processing_error = SettlementProcessingError(
                "Settlement service failed."
            )

            processing_error.__cause__ = error

            failed_settlements.append(
                {
                    "reference": settlement.get(
                        "reference",
                        "UNKNOWN"
                    ),
                    "error": str(processing_error),
                    "cause": str(processing_error.__cause__)
                }
            )

        except InvalidSettlementError as error:
            # Business-rule failure
            failed_settlements.append(
                {
                    "reference": settlement.get(
                        "reference",
                        "UNKNOWN"
                    )
                    if isinstance(settlement, dict)
                    else "UNKNOWN",
                    "error": str(error),
                    "cause": None
                }
            )

    return (
        balance,
        successful_settlements,
        failed_settlements
    )


# Sample banking case
account_number = "ACC-1077"
balance = 10000

settlements = [
    {"reference": "TXN-001", "amount": 2000},
    {"reference": "TXN-002", "amount": "abc"},
    {"reference": "TXN-003", "amount": 3000},
    {"reference": "TXN-004", "amount": 4000},
    {"reference": "TXN-005", "amount": -500}
]

try:
    (
        final_balance,
        successful_settlements,
        failed_settlements
    ) = process_settlement_batch(
        account_number,
        balance,
        settlements
    )

    print("\nAccount Number:", account_number)
    print("Final Balance:", final_balance)

    print("\nSuccessful Settlements:")
    for reference in successful_settlements:
        print(reference)

    print("\nFailed Settlements:")
    for settlement in failed_settlements:
        print(
            settlement["reference"],
            "-",
            settlement["error"]
        )

        if settlement["cause"]:
            print("  Original Cause:", settlement["cause"])

except SettlementError as error:
    print("Settlement batch failed:", error)


Account Number: ACC-1077
Final Balance: 4000

Successful Settlements:
TXN-001
TXN-004

Failed Settlements:
TXN-002 - Settlement amount must be a number.
TXN-003 - Settlement service failed.
  Original Cause: Settlement service temporarily unavailable.
TXN-005 - Settlement amount must be greater than zero.


## === Q78. Report exact paths in nested statements for banking case 1; use sample reference ACC-1078 and explain the result. ===

In [28]:
class PaymentValidationError(Exception):
    """Raised when payment validation fails."""
    pass


class TransactionProcessingError(Exception):
    """Raised when transaction processing fails."""
    pass


def validate_payment(amount):
    """Validate the payment amount."""

    if not isinstance(amount, (int, float)):
        raise TypeError(
            "Payment amount must be a number."
        )

    if amount <= 0:
        raise ValueError(
            "Payment amount must be greater than zero."
        )

    return True


def process_payment(account_number, balance, amount):
    """Process a payment using nested exception handling."""

    try:
        # Outer transaction processing
        try:
            # Smallest risky statement
            validate_payment(amount)

        except (TypeError, ValueError) as error:
            # Preserve the exact original cause
            raise PaymentValidationError(
                "Payment validation failed."
            ) from error

        # Business rule check
        if amount > balance:
            raise PaymentValidationError(
                "Insufficient funds."
            )

        # Update balance only after all checks succeed
        balance -= amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "amount": amount,
            "remaining_balance": balance
        }

    except PaymentValidationError as error:
        # Outer layer handles the business failure
        raise TransactionProcessingError(
            "Transaction processing failed."
        ) from error


# Sample banking case
account_number = "ACC-1078"
balance = 10000
payment_amount = "abc"

try:
    result = process_payment(
        account_number,
        balance,
        payment_amount
    )

    print("Payment successful.")
    print(result)

except TransactionProcessingError as error:
    print("Transaction failed:", error)
    print("Immediate cause:", error.__cause__)
    print("Original cause:", error.__cause__.__cause__)
    print("Account Number:", account_number)
    print("Account Balance:", balance)

Transaction failed: Transaction processing failed.
Immediate cause: Payment validation failed.
Original cause: Payment amount must be a number.
Account Number: ACC-1078
Account Balance: 10000


## === Q79. Design idempotent recovery after timeout for banking case 1; use sample reference ACC-1079 and explain the result. ===

In [29]:
class PaymentTimeoutError(Exception):
    """Raised when the payment service times out."""
    pass


class InvalidPaymentError(Exception):
    """Raised when payment details are invalid."""
    pass


class PaymentProcessingError(Exception):
    """Raised when payment processing fails."""
    pass


# Store successfully processed transaction references
processed_transactions = set()


def payment_service(transaction_reference, amount):
    """
    Simulate a payment service.

    The first attempt for TXN-001 times out,
    but the payment is actually processed.
    """

    if transaction_reference == "TXN-001":
        if transaction_reference not in processed_transactions:
            # Simulate timeout after payment was processed
            processed_transactions.add(transaction_reference)

            raise PaymentTimeoutError(
                "Payment service timed out after processing the payment."
            )

    return True


def process_payment(account_number, balance, payment):
    """Process a payment safely using idempotent recovery."""

    try:
        # Validate payment structure
        if not isinstance(payment, dict):
            raise InvalidPaymentError(
                "Payment must be a dictionary."
            )

        transaction_reference = payment.get("reference")
        amount = payment.get("amount")

        if not transaction_reference:
            raise InvalidPaymentError(
                "Transaction reference is required."
            )

        if not isinstance(amount, (int, float)):
            raise InvalidPaymentError(
                "Payment amount must be a number."
            )

        if amount <= 0:
            raise InvalidPaymentError(
                "Payment amount must be greater than zero."
            )

        if amount > balance:
            raise InvalidPaymentError(
                "Insufficient funds."
            )

        # Check idempotency before attempting the payment
        if transaction_reference in processed_transactions:
            print("Transaction already processed.")
            print("No duplicate balance update performed.")

            return {
                "status": "ALREADY_PROCESSED",
                "account_number": account_number,
                "reference": transaction_reference,
                "remaining_balance": balance
            }

        try:
            # Smallest risky statement
            payment_service(
                transaction_reference,
                amount
            )

        except PaymentTimeoutError as error:
            # The service may have completed the payment
            # even though the response timed out.
            if transaction_reference in processed_transactions:
                print("Timeout detected.")
                print("Payment was already processed by the service.")

                # Apply the balance update exactly once
                balance -= amount

                return {
                    "status": "RECOVERED",
                    "account_number": account_number,
                    "reference": transaction_reference,
                    "remaining_balance": balance
                }

            raise PaymentProcessingError(
                "Payment timed out and could not be confirmed."
            ) from error

        # Normal successful payment
        if transaction_reference not in processed_transactions:
            processed_transactions.add(
                transaction_reference
            )

        balance -= amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "reference": transaction_reference,
            "remaining_balance": balance
        }

    except InvalidPaymentError:
        raise


# Sample banking case
account_number = "ACC-1079"
balance = 10000

payment = {
    "reference": "TXN-001",
    "amount": 3000
}

try:
    result = process_payment(
        account_number,
        balance,
        payment
    )

    print("\nPayment Result:")
    print("Status:", result["status"])
    print("Account Number:", result["account_number"])
    print("Transaction Reference:", result["reference"])
    print("Remaining Balance:", result["remaining_balance"])

except InvalidPaymentError as error:
    print("Payment rejected:", error)

except PaymentProcessingError as error:
    print("Payment failed:", error)
    print("Original cause:", error.__cause__)

Timeout detected.
Payment was already processed by the service.

Payment Result:
Status: RECOVERED
Account Number: ACC-1079
Transaction Reference: TXN-001
Remaining Balance: 7000


## === Q80. Build an exception-safe reconciliation pipeline for banking case 1; use sample reference ACC-1080 and explain the result. ===

In [30]:
class ReconciliationError(Exception):
    """Base class for reconciliation errors."""
    pass


class InvalidRecordError(ReconciliationError):
    """Raised when a reconciliation record is invalid."""
    pass


class ExternalServiceError(ReconciliationError):
    """Raised when the external reconciliation service fails."""
    pass


class ReconciliationProcessingError(ReconciliationError):
    """Raised when reconciliation processing fails."""
    pass


def get_external_amount(record):
    """
    Simulate retrieving the amount from an external
    settlement system.

    TXN-003 simulates an external service failure.
    """

    if record["reference"] == "TXN-003":
        raise ExternalServiceError(
            "External settlement service unavailable."
        )

    return record["external_amount"]


def reconcile_record(record):
    """Reconcile one banking transaction."""

    try:
        # Validate record structure
        if not isinstance(record, dict):
            raise InvalidRecordError(
                "Reconciliation record must be a dictionary."
            )

        reference = record.get("reference")
        bank_amount = record.get("bank_amount")

        if not reference:
            raise InvalidRecordError(
                "Transaction reference is required."
            )

        if not isinstance(bank_amount, (int, float)):
            raise InvalidRecordError(
                "Bank amount must be a number."
            )

        if bank_amount < 0:
            raise InvalidRecordError(
                "Bank amount cannot be negative."
            )

        try:
            # Smallest risky statement
            external_amount = get_external_amount(record)

        except ExternalServiceError as error:
            raise ReconciliationProcessingError(
                "Unable to retrieve external settlement amount."
            ) from error

        # Validate external amount
        if not isinstance(external_amount, (int, float)):
            raise InvalidRecordError(
                "External amount must be a number."
            )

        # Compare the two amounts
        if bank_amount == external_amount:
            return {
                "reference": reference,
                "status": "RECONCILED",
                "bank_amount": bank_amount,
                "external_amount": external_amount
            }

        return {
            "reference": reference,
            "status": "MISMATCH",
            "bank_amount": bank_amount,
            "external_amount": external_amount
        }

    except ReconciliationError:
        raise


def reconciliation_pipeline(
    account_number,
    balance,
    records
):
    """Run reconciliation without changing the account balance."""

    reconciled = []
    mismatches = []
    failed_records = []

    for record in records:
        try:
            result = reconcile_record(record)

            if result["status"] == "RECONCILED":
                reconciled.append(result)

            else:
                mismatches.append(result)

        except ReconciliationProcessingError as error:
            failed_records.append(
                {
                    "reference": record.get(
                        "reference",
                        "UNKNOWN"
                    )
                    if isinstance(record, dict)
                    else "UNKNOWN",
                    "error": str(error),
                    "cause": str(error.__cause__)
                }
            )

        except InvalidRecordError as error:
            failed_records.append(
                {
                    "reference": record.get(
                        "reference",
                        "UNKNOWN"
                    )
                    if isinstance(record, dict)
                    else "UNKNOWN",
                    "error": str(error),
                    "cause": None
                }
            )

    return (
        account_number,
        balance,
        reconciled,
        mismatches,
        failed_records
    )


# Sample banking case
account_number = "ACC-1080"
balance = 10000

records = [
    {
        "reference": "TXN-001",
        "bank_amount": 2000,
        "external_amount": 2000
    },
    {
        "reference": "TXN-002",
        "bank_amount": 3000,
        "external_amount": 2500
    },
    {
        "reference": "TXN-003",
        "bank_amount": 1500,
        "external_amount": 1500
    },
    {
        "reference": "TXN-004",
        "bank_amount": "abc",
        "external_amount": 1000
    },
    {
        "reference": "TXN-005",
        "bank_amount": 1000,
        "external_amount": 1000
    }
]

try:
    (
        account_number,
        final_balance,
        reconciled,
        mismatches,
        failed_records
    ) = reconciliation_pipeline(
        account_number,
        balance,
        records
    )

    print("Account Number:", account_number)
    print("Account Balance:", final_balance)

    print("\nReconciled Transactions:")
    for record in reconciled:
        print(
            record["reference"],
            "-",
            record["bank_amount"]
        )

    print("\nMismatched Transactions:")
    for record in mismatches:
        print(
            record["reference"],
            "- Bank:",
            record["bank_amount"],
            "| External:",
            record["external_amount"]
        )

    print("\nFailed Records:")
    for record in failed_records:
        print(
            record["reference"],
            "-",
            record["error"]
        )

        if record["cause"]:
            print("  Original Cause:", record["cause"])

except ReconciliationError as error:
    print("Reconciliation failed:", error)

Account Number: ACC-1080
Account Balance: 10000

Reconciled Transactions:
TXN-001 - 2000
TXN-005 - 1000

Mismatched Transactions:
TXN-002 - Bank: 3000 | External: 2500

Failed Records:
TXN-003 - Unable to retrieve external settlement amount.
  Original Cause: External settlement service unavailable.
TXN-004 - Bank amount must be a number.


## === Q81. Design a banking exception hierarchy for banking case 2; use sample reference ACC-1081 and explain the result. ===

In [32]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class TransactionError(BankingError):
    """Base class for transaction-related errors."""
    pass


class InsufficientFundsError(TransactionError):
    """Raised when the account has insufficient funds."""
    pass


class InvalidTransactionError(TransactionError):
    """Raised when a transaction is invalid."""
    pass


class PaymentGatewayError(BankingError):
    """Raised when the payment gateway fails."""
    pass


class TransactionProcessingError(BankingError):
    """Raised when transaction processing fails."""
    pass


def process_withdrawal(account_number, balance, amount):
    try:
        # Validate the transaction amount
        if not isinstance(amount, (int, float)):
            raise InvalidTransactionError(
                "Withdrawal amount must be a number."
            )

        if amount <= 0:
            raise InvalidTransactionError(
                "Withdrawal amount must be greater than zero."
            )

        # Check available balance
        if amount > balance:
            raise InsufficientFundsError(
                "Insufficient funds for this withdrawal."
            )

        # Update balance only after validation succeeds
        balance -= amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "withdrawal_amount": amount,
            "remaining_balance": balance
        }

    except (InvalidTransactionError, InsufficientFundsError):
        # Business-rule errors are passed to the caller
        raise

    except Exception as error:
        # Unexpected errors are wrapped while preserving the cause
        raise TransactionProcessingError(
            "Unexpected error while processing withdrawal."
        ) from error


# Banking case
account_number = "ACC-1081"
balance = 8000
withdrawal_amount = 10000

try:
    result = process_withdrawal(
        account_number,
        balance,
        withdrawal_amount
    )

    print("Withdrawal successful.")
    print("Account Number:", result["account_number"])
    print("Withdrawal Amount:", result["withdrawal_amount"])
    print("Remaining Balance:", result["remaining_balance"])

except InsufficientFundsError as error:
    print("Withdrawal rejected:", error)
    print("Account Number:", account_number)
    print("Account Balance:", balance)

except InvalidTransactionError as error:
    print("Invalid transaction:", error)

except TransactionProcessingError as error:
    print("Transaction processing failed:", error)
    print("Original Cause:", error.__cause__)

except BankingError as error:
    print("Banking error:", error)

Withdrawal rejected: Insufficient funds for this withdrawal.
Account Number: ACC-1081
Account Balance: 8000


## === Q82. Make a transfer rollback-safe for banking case 2; use sample reference ACC-1082 and explain the result. ===

In [33]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class TransferError(BankingError):
    """Base class for transfer-related errors."""
    pass


class InvalidTransferError(TransferError):
    """Raised when transfer details are invalid."""
    pass


class InsufficientFundsError(TransferError):
    """Raised when the source account has insufficient funds."""
    pass


class BeneficiaryUpdateError(TransferError):
    """Raised when the beneficiary account update fails."""
    pass


class TransferProcessingError(BankingError):
    """Raised when an unexpected transfer error occurs."""
    pass


def update_beneficiary(balance, amount):
    """
    Simulate updating the beneficiary account.

    This function raises an error to demonstrate
    rollback protection.
    """
    raise BeneficiaryUpdateError(
        "Beneficiary account update failed."
    )


def process_transfer(
    account_number,
    beneficiary_account,
    source_balance,
    beneficiary_balance,
    amount
):
    # Save original balances before making any changes
    original_source_balance = source_balance
    original_beneficiary_balance = beneficiary_balance

    try:
        # Validate transfer amount
        if not isinstance(amount, (int, float)):
            raise InvalidTransferError(
                "Transfer amount must be a number."
            )

        if amount <= 0:
            raise InvalidTransferError(
                "Transfer amount must be greater than zero."
            )

        # Check source account balance
        if amount > source_balance:
            raise InsufficientFundsError(
                "Insufficient funds in source account."
            )

        # Deduct from source account
        source_balance -= amount

        try:
            # Update beneficiary account
            beneficiary_balance = update_beneficiary(
                beneficiary_balance,
                amount
            )

        except BeneficiaryUpdateError as error:
            raise TransferProcessingError(
                "Transfer failed while updating beneficiary account."
            ) from error

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "beneficiary_account": beneficiary_account,
            "transfer_amount": amount,
            "source_balance": source_balance,
            "beneficiary_balance": beneficiary_balance
        }

    except (InvalidTransferError, InsufficientFundsError):
        # Business-rule failures
        raise

    except TransferProcessingError:
        # Preserve the processing failure
        # and roll back both balances
        source_balance = original_source_balance
        beneficiary_balance = original_beneficiary_balance

        raise

    except Exception as error:
        # Unexpected error: roll back and preserve the cause
        source_balance = original_source_balance
        beneficiary_balance = original_beneficiary_balance

        raise TransferProcessingError(
            "Unexpected error while processing transfer."
        ) from error


# Banking case 2
account_number = "ACC-1082"
beneficiary_account = "BEN-1002"

source_balance = 10000
beneficiary_balance = 3000
transfer_amount = 4000

original_source_balance = source_balance
original_beneficiary_balance = beneficiary_balance

try:
    result = process_transfer(
        account_number,
        beneficiary_account,
        source_balance,
        beneficiary_balance,
        transfer_amount
    )

    print("Transfer successful.")
    print("Account Number:", result["account_number"])
    print("Beneficiary Account:", result["beneficiary_account"])
    print("Transfer Amount:", result["transfer_amount"])
    print("Source Balance:", result["source_balance"])
    print("Beneficiary Balance:", result["beneficiary_balance"])

except InvalidTransferError as error:
    print("Invalid transfer:", error)

except InsufficientFundsError as error:
    print("Transfer rejected:", error)

except TransferProcessingError as error:
    # Restore balances at the caller level as well,
    # because integers are passed by value into the function.
    source_balance = original_source_balance
    beneficiary_balance = original_beneficiary_balance

    print("Transfer failed:", error)
    print("Original Cause:", error.__cause__)

    print("\nRollback completed.")
    print("Account Number:", account_number)
    print("Source Balance:", source_balance)
    print("Beneficiary Account:", beneficiary_account)
    print("Beneficiary Balance:", beneficiary_balance)

Transfer failed: Transfer failed while updating beneficiary account.
Original Cause: Beneficiary account update failed.

Rollback completed.
Account Number: ACC-1082
Source Balance: 10000
Beneficiary Account: BEN-1002
Beneficiary Balance: 3000


## === Q83. Preserve causes across service layers for banking case 2; use sample reference ACC-1083 and explain the result. ===

In [34]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class PaymentGatewayError(BankingError):
    """Raised when the payment gateway fails."""
    pass


class PaymentProcessingError(BankingError):
    """Raised when payment processing fails."""
    pass


def payment_gateway(payment_amount):
    """
    Simulate a payment gateway.

    The gateway fails because of a connection problem.
    """

    if not isinstance(payment_amount, (int, float)):
        raise TypeError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise ValueError(
            "Payment amount must be greater than zero."
        )

    # Simulate infrastructure failure
    raise ConnectionError(
        "Payment gateway connection failed."
    )


def process_payment(
    account_number,
    balance,
    payment_amount
):
    try:
        # Smallest risky statement
        payment_gateway(payment_amount)

    except (TypeError, ValueError) as error:
        # Business/input validation failure
        raise PaymentGatewayError(
            "Invalid payment information."
        ) from error

    except ConnectionError as error:
        # Infrastructure failure
        raise PaymentGatewayError(
            "Payment gateway is unavailable."
        ) from error

    # Balance is updated only after gateway success
    balance -= payment_amount

    return {
        "status": "SUCCESS",
        "account_number": account_number,
        "payment_amount": payment_amount,
        "remaining_balance": balance
    }


def banking_service(
    account_number,
    balance,
    payment_amount
):
    try:
        result = process_payment(
            account_number,
            balance,
            payment_amount
        )

        return result

    except PaymentGatewayError as error:
        raise PaymentProcessingError(
            "Payment processing failed."
        ) from error


# Banking case 2
account_number = "ACC-1083"
balance = 10000
payment_amount = 3000

try:
    result = banking_service(
        account_number,
        balance,
        payment_amount
    )

    print("Payment successful.")
    print("Account Number:", result["account_number"])
    print("Payment Amount:", result["payment_amount"])
    print("Remaining Balance:", result["remaining_balance"])

except PaymentProcessingError as error:
    print("Payment failed:", error)
    print("Immediate Cause:", error.__cause__)

    if error.__cause__:
        print(
            "Original Cause:",
            error.__cause__.__cause__
        )

    print("Account Number:", account_number)
    print("Account Balance:", balance)

Payment failed: Payment processing failed.
Immediate Cause: Payment gateway is unavailable.
Original Cause: Payment gateway connection failed.
Account Number: ACC-1083
Account Balance: 10000


## === Q84. Implement bounded exponential retry for banking case 2; use sample reference ACC-1084 and explain the result. ===

In [35]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class InvalidPaymentError(BankingError):
    """Raised when payment details are invalid."""
    pass


class InsufficientFundsError(BankingError):
    """Raised when the account has insufficient funds."""
    pass


class TransientServiceError(BankingError):
    """Raised when a service temporarily fails."""
    pass


class PaymentProcessingError(BankingError):
    """Raised when payment processing fails after retries."""
    pass


def payment_service():
    """
    Simulate a temporary payment service failure.

    The service fails for the first three attempts
    and succeeds on the fourth attempt.
    """
    payment_service.attempts += 1

    if payment_service.attempts <= 3:
        raise TransientServiceError(
            "Payment service temporarily unavailable."
        )

    return True


payment_service.attempts = 0


def process_payment(
    account_number,
    balance,
    payment_amount,
    max_retries=4,
    base_delay=1,
    max_delay=4
):
    # Validate payment amount
    if not isinstance(payment_amount, (int, float)):
        raise InvalidPaymentError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise InvalidPaymentError(
            "Payment amount must be greater than zero."
        )

    # Check balance before attempting payment
    if payment_amount > balance:
        raise InsufficientFundsError(
            "Insufficient funds."
        )

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            # Smallest risky statement
            payment_service()

            # Update balance only after successful service call
            balance -= payment_amount

            return {
                "status": "SUCCESS",
                "account_number": account_number,
                "payment_amount": payment_amount,
                "remaining_balance": balance,
                "attempts": attempt
            }

        except TransientServiceError as error:
            last_error = error

            # Exponential backoff with an upper limit
            delay = min(
                base_delay * (2 ** (attempt - 1)),
                max_delay
            )

            print(
                f"Attempt {attempt} failed. "
                f"Retrying after {delay} second(s)."
            )

            if attempt == max_retries:
                raise PaymentProcessingError(
                    "Payment failed after all retry attempts."
                ) from last_error


# Banking case 2
account_number = "ACC-1084"
balance = 10000
payment_amount = 3000

try:
    result = process_payment(
        account_number,
        balance,
        payment_amount
    )

    print("\nPayment successful.")
    print("Account Number:", result["account_number"])
    print("Payment Amount:", result["payment_amount"])
    print("Remaining Balance:", result["remaining_balance"])
    print("Attempts Used:", result["attempts"])

except InvalidPaymentError as error:
    print("\nInvalid payment:", error)

except InsufficientFundsError as error:
    print("\nPayment rejected:", error)

except PaymentProcessingError as error:
    print("\nPayment failed:", error)
    print("Original Cause:", error.__cause__)

Attempt 1 failed. Retrying after 1 second(s).
Attempt 2 failed. Retrying after 2 second(s).
Attempt 3 failed. Retrying after 4 second(s).

Payment successful.
Account Number: ACC-1084
Payment Amount: 3000
Remaining Balance: 7000
Attempts Used: 4


## === Q85. Create a transaction context manager for banking case 2; use sample reference ACC-1085 and explain the result. ===

In [36]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class TransactionError(BankingError):
    """Raised when a transaction operation fails."""
    pass


class InvalidAmountError(TransactionError):
    """Raised when an amount is invalid."""
    pass


class InsufficientFundsError(TransactionError):
    """Raised when there are insufficient funds."""
    pass


class TransactionContext:
    """Context manager for a rollback-safe banking transaction."""

    def __init__(self, balance):
        self.original_balance = balance
        self.balance = balance

    def __enter__(self):
        print("Transaction started.")
        return self

    def withdraw(self, amount):
        """Withdraw money from the account."""

        if not isinstance(amount, (int, float)):
            raise InvalidAmountError(
                "Withdrawal amount must be a number."
            )

        if amount <= 0:
            raise InvalidAmountError(
                "Withdrawal amount must be greater than zero."
            )

        if amount > self.balance:
            raise InsufficientFundsError(
                "Insufficient funds."
            )

        self.balance -= amount

        print("Withdrawal completed:", amount)

    def deposit(self, amount):
        """Deposit money into the account."""

        if not isinstance(amount, (int, float)):
            raise InvalidAmountError(
                "Deposit amount must be a number."
            )

        if amount <= 0:
            raise InvalidAmountError(
                "Deposit amount must be greater than zero."
            )

        self.balance += amount

        print("Deposit completed:", amount)

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            print("Transaction committed.")
            return False

        # Roll back to the original balance
        self.balance = self.original_balance

        print("Transaction rolled back.")
        print("Reason:", exc_value)

        # Do not hide the original exception
        return False


# Banking case 2
account_number = "ACC-1085"
starting_balance = 10000

try:
    transaction = TransactionContext(starting_balance)

    with transaction:
        transaction.withdraw(3000)
        transaction.deposit(1000)

    print("\nTransaction successful.")
    print("Account Number:", account_number)
    print("Original Balance:", starting_balance)
    print("Final Balance:", transaction.balance)

except InvalidAmountError as error:
    print("\nInvalid transaction:", error)

except InsufficientFundsError as error:
    print("\nTransaction rejected:", error)

except BankingError as error:
    print("\nBanking error:", error)

Transaction started.
Withdrawal completed: 3000
Deposit completed: 1000
Transaction committed.

Transaction successful.
Account Number: ACC-1085
Original Balance: 10000
Final Balance: 8000


## === Q86. Build a dead-letter queue for payment events for banking case 2; use sample reference ACC-1086 and explain the result. ===

In [37]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class InvalidPaymentError(BankingError):
    """Raised when a payment event is invalid."""
    pass


class TransientPaymentError(BankingError):
    """Raised when payment processing temporarily fails."""
    pass


class PaymentProcessingError(BankingError):
    """Raised when payment processing fails after retries."""
    pass


def payment_service(event):
    """
    Simulate an external payment service.

    TXN-003 fails temporarily on every attempt
    to demonstrate the dead-letter queue.
    """

    if event["reference"] == "TXN-003":
        raise TransientPaymentError(
            "Payment service temporarily unavailable."
        )

    return True


def process_payment_events(
    account_number,
    balance,
    payment_events,
    max_retries=2
):
    successful_payments = []
    dead_letter_queue = []

    for event in payment_events:
        try:
            # Validate event structure
            if not isinstance(event, dict):
                raise InvalidPaymentError(
                    "Payment event must be a dictionary."
                )

            reference = event.get("reference")
            amount = event.get("amount")

            if not reference:
                raise InvalidPaymentError(
                    "Payment reference is required."
                )

            if not isinstance(amount, (int, float)):
                raise InvalidPaymentError(
                    "Payment amount must be a number."
                )

            if amount <= 0:
                raise InvalidPaymentError(
                    "Payment amount must be greater than zero."
                )

            if amount > balance:
                raise InvalidPaymentError(
                    "Insufficient funds."
                )

            payment_succeeded = False
            last_error = None

            # Retry temporary service failures
            for attempt in range(1, max_retries + 1):
                try:
                    # Smallest risky statement
                    payment_service(event)

                    payment_succeeded = True
                    break

                except TransientPaymentError as error:
                    last_error = error

                    print(
                        f"{reference}: "
                        f"Attempt {attempt} failed - {error}"
                    )

            # Move event to DLQ if all retries fail
            if not payment_succeeded:
                raise PaymentProcessingError(
                    "Payment failed after all retry attempts."
                ) from last_error

            # Update balance only after successful processing
            balance -= amount
            successful_payments.append(reference)

        except (
            InvalidPaymentError,
            PaymentProcessingError
        ) as error:

            dead_letter_queue.append(
                {
                    "reference": (
                        event.get("reference", "UNKNOWN")
                        if isinstance(event, dict)
                        else "UNKNOWN"
                    ),
                    "error": str(error),
                    "cause": (
                        str(error.__cause__)
                        if error.__cause__
                        else None
                    )
                }
            )

    return (
        balance,
        successful_payments,
        dead_letter_queue
    )


# Banking case 2
account_number = "ACC-1086"
balance = 10000

payment_events = [
    {"reference": "TXN-001", "amount": 2000},
    {"reference": "TXN-002", "amount": "abc"},
    {"reference": "TXN-003", "amount": 3000},
    {"reference": "TXN-004", "amount": 4000},
    {"reference": "TXN-005", "amount": -500}
]

try:
    (
        final_balance,
        successful_payments,
        dead_letter_queue
    ) = process_payment_events(
        account_number,
        balance,
        payment_events
    )

    print("\nAccount Number:", account_number)
    print("Final Balance:", final_balance)

    print("\nSuccessful Payments:")
    for reference in successful_payments:
        print(reference)

    print("\nDead-Letter Queue:")

    for event in dead_letter_queue:
        print(
            event["reference"],
            "-",
            event["error"]
        )

        if event["cause"]:
            print("  Original Cause:", event["cause"])

except BankingError as error:
    print("Payment processing failed:", error)

TXN-003: Attempt 1 failed - Payment service temporarily unavailable.
TXN-003: Attempt 2 failed - Payment service temporarily unavailable.

Account Number: ACC-1086
Final Balance: 4000

Successful Payments:
TXN-001
TXN-004

Dead-Letter Queue:
TXN-002 - Payment amount must be a number.
TXN-003 - Payment failed after all retry attempts.
  Original Cause: Payment service temporarily unavailable.
TXN-005 - Payment amount must be greater than zero.


## === Q87. Make batch settlement failure-isolated for banking case 2; use sample reference ACC-1087 and explain the result. ===

In [38]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class InvalidSettlementError(BankingError):
    """Raised when settlement details are invalid."""
    pass


class SettlementServiceError(BankingError):
    """Raised when the settlement service fails."""
    pass


class SettlementProcessingError(BankingError):
    """Raised when settlement processing fails."""
    pass


def settlement_service(transaction):
    """
    Simulate an external settlement service.

    TXN-003 fails to demonstrate failure isolation.
    """
    if transaction["reference"] == "TXN-003":
        raise SettlementServiceError(
            "Settlement service temporarily unavailable."
        )

    return True


def process_batch_settlement(
    account_number,
    balance,
    settlement_events
):
    successful_settlements = []
    failed_settlements = []

    for event in settlement_events:

        try:
            # Validate settlement event
            if not isinstance(event, dict):
                raise InvalidSettlementError(
                    "Settlement event must be a dictionary."
                )

            reference = event.get("reference")
            amount = event.get("amount")

            if not reference:
                raise InvalidSettlementError(
                    "Settlement reference is required."
                )

            if not isinstance(amount, (int, float)):
                raise InvalidSettlementError(
                    "Settlement amount must be a number."
                )

            if amount <= 0:
                raise InvalidSettlementError(
                    "Settlement amount must be greater than zero."
                )

            if amount > balance:
                raise InvalidSettlementError(
                    "Insufficient funds for settlement."
                )

            # Risky statement: external settlement service
            try:
                settlement_service(event)

            except SettlementServiceError as error:
                raise SettlementProcessingError(
                    "Settlement failed because the service was unavailable."
                ) from error

            # Update balance only after successful settlement
            balance -= amount

            successful_settlements.append(
                {
                    "reference": reference,
                    "amount": amount
                }
            )

        except (
            InvalidSettlementError,
            SettlementProcessingError
        ) as error:

            failed_settlements.append(
                {
                    "reference": (
                        event.get("reference", "UNKNOWN")
                        if isinstance(event, dict)
                        else "UNKNOWN"
                    ),
                    "amount": (
                        event.get("amount", "UNKNOWN")
                        if isinstance(event, dict)
                        else "UNKNOWN"
                    ),
                    "error": str(error),
                    "cause": (
                        str(error.__cause__)
                        if error.__cause__
                        else None
                    )
                }
            )

            # Continue with the next settlement
            continue

    return (
        balance,
        successful_settlements,
        failed_settlements
    )


# Sample banking case
account_number = "ACC-1087"
balance = 10000

settlement_events = [
    {"reference": "TXN-001", "amount": 2000},
    {"reference": "TXN-002", "amount": "abc"},
    {"reference": "TXN-003", "amount": 3000},
    {"reference": "TXN-004", "amount": 4000},
    {"reference": "TXN-005", "amount": -500}
]


try:
    (
        final_balance,
        successful_settlements,
        failed_settlements
    ) = process_batch_settlement(
        account_number,
        balance,
        settlement_events
    )

    print("Account Number:", account_number)
    print("Final Balance:", final_balance)

    print("\nSuccessful Settlements:")

    for settlement in successful_settlements:
        print(
            settlement["reference"],
            "-",
            settlement["amount"]
        )

    print("\nFailed Settlements:")

    for settlement in failed_settlements:
        print(
            settlement["reference"],
            "-",
            settlement["error"]
        )

        if settlement["cause"]:
            print(
                "  Original Cause:",
                settlement["cause"]
            )

except BankingError as error:
    print("Batch settlement failed:", error)

Account Number: ACC-1087
Final Balance: 4000

Successful Settlements:
TXN-001 - 2000
TXN-004 - 4000

Failed Settlements:
TXN-002 - Settlement amount must be a number.
TXN-003 - Settlement failed because the service was unavailable.
  Original Cause: Settlement service temporarily unavailable.
TXN-005 - Settlement amount must be greater than zero.


## === Q88. Report exact paths in nested statements for banking case 2; use sample reference ACC-1088 and explain the result. ===

In [39]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class PaymentValidationError(BankingError):
    """Raised when payment validation fails."""
    pass


class TransactionProcessingError(BankingError):
    """Raised when transaction processing fails."""
    pass


def validate_payment(payment_amount):
    """Validate the payment amount."""

    if not isinstance(payment_amount, (int, float)):
        raise TypeError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise ValueError(
            "Payment amount must be greater than zero."
        )

    return True


def process_payment(
    account_number,
    balance,
    payment_amount
):
    try:
        # Smallest risky statement
        try:
            validate_payment(payment_amount)

        except (TypeError, ValueError) as error:
            raise PaymentValidationError(
                "Payment validation failed."
            ) from error

        # Update balance only after validation succeeds
        balance -= payment_amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "payment_amount": payment_amount,
            "remaining_balance": balance
        }

    except PaymentValidationError as error:
        raise TransactionProcessingError(
            "Transaction processing failed."
        ) from error


# Sample banking case
account_number = "ACC-1088"
balance = 10000
payment_amount = "abc"


try:
    result = process_payment(
        account_number,
        balance,
        payment_amount
    )

    print("Payment successful.")
    print("Account Number:", result["account_number"])
    print("Payment Amount:", result["payment_amount"])
    print("Remaining Balance:", result["remaining_balance"])

except TransactionProcessingError as error:
    print("Transaction failed:", error)
    print("Immediate Cause:", error.__cause__)

    if error.__cause__:
        print(
            "Original Cause:",
            error.__cause__.__cause__
        )

    print("Account Number:", account_number)
    print("Account Balance:", balance)

except BankingError as error:
    print("Banking error:", error)

Transaction failed: Transaction processing failed.
Immediate Cause: Payment validation failed.
Original Cause: Payment amount must be a number.
Account Number: ACC-1088
Account Balance: 10000


## === Q89. Design idempotent recovery after timeout for banking case 2; use sample reference ACC-1089 and explain the result. ===

In [40]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class InvalidPaymentError(BankingError):
    """Raised when payment details are invalid."""
    pass


class PaymentTimeoutError(BankingError):
    """Raised when the payment service times out."""
    pass


class PaymentProcessingError(BankingError):
    """Raised when payment processing fails."""
    pass


processed_transactions = set()


def payment_service(transaction_reference):
    """
    Simulate an external payment service.

    The first attempt for TXN-001 completes the payment
    but raises a timeout before the response reaches us.
    """

    if transaction_reference == "TXN-001":
        if transaction_reference not in processed_transactions:
            processed_transactions.add(transaction_reference)

            raise PaymentTimeoutError(
                "Payment completed, but the response timed out."
            )

    return True


def process_payment(
    account_number,
    balance,
    transaction_reference,
    payment_amount
):
    # Validate input before processing
    if not transaction_reference:
        raise InvalidPaymentError(
            "Transaction reference is required."
        )

    if not isinstance(payment_amount, (int, float)):
        raise InvalidPaymentError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise InvalidPaymentError(
            "Payment amount must be greater than zero."
        )

    if payment_amount > balance:
        raise InvalidPaymentError(
            "Insufficient funds."
        )

    # Idempotency check
    if transaction_reference in processed_transactions:
        return {
            "status": "ALREADY_PROCESSED",
            "account_number": account_number,
            "transaction_reference": transaction_reference,
            "remaining_balance": balance
        }

    try:
        # Smallest risky statement
        payment_service(transaction_reference)

    except PaymentTimeoutError as error:
        raise PaymentProcessingError(
            "Payment response timed out."
        ) from error

    # Mark transaction as processed only after success
    processed_transactions.add(transaction_reference)

    balance -= payment_amount

    return {
        "status": "SUCCESS",
        "account_number": account_number,
        "transaction_reference": transaction_reference,
        "payment_amount": payment_amount,
        "remaining_balance": balance
    }


# Sample banking case
account_number = "ACC-1089"
balance = 10000
transaction_reference = "TXN-001"
payment_amount = 3000


# First attempt
try:
    result = process_payment(
        account_number,
        balance,
        transaction_reference,
        payment_amount
    )

    balance = result["remaining_balance"]

except PaymentProcessingError as error:
    print("Timeout detected.")
    print("Original Cause:", error.__cause__)

    # Recover by checking whether the transaction
    # was already processed by the payment service.
    if transaction_reference in processed_transactions:
        print("Payment was already processed by the service.")

        balance -= payment_amount

        result = {
            "status": "RECOVERED",
            "account_number": account_number,
            "transaction_reference": transaction_reference,
            "remaining_balance": balance
        }

    else:
        print("Payment was not processed.")
        result = None


# Final result
if result:
    print("\nPayment Result:")
    print("Status:", result["status"])
    print("Account Number:", result["account_number"])
    print(
        "Transaction Reference:",
        result["transaction_reference"]
    )
    print("Remaining Balance:", result["remaining_balance"])

Timeout detected.
Original Cause: Payment completed, but the response timed out.
Payment was already processed by the service.

Payment Result:
Status: RECOVERED
Account Number: ACC-1089
Transaction Reference: TXN-001
Remaining Balance: 7000


## === Q90. Build an exception-safe reconciliation pipeline for banking case 2; use sample reference ACC-1090 and explain the result. ===

In [41]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class ReconciliationError(BankingError):
    """Base class for reconciliation errors."""
    pass


class InvalidRecordError(ReconciliationError):
    """Raised when a reconciliation record is invalid."""
    pass


class ExternalServiceError(ReconciliationError):
    """Raised when the external settlement service fails."""
    pass


class ReconciliationProcessingError(ReconciliationError):
    """Raised when reconciliation processing fails."""
    pass


def get_external_amount(transaction_reference):
    """
    Simulate an external settlement service.
    TXN-003 represents a realistic service failure.
    """

    external_amounts = {
        "TXN-001": 2000,
        "TXN-002": 2500,
        "TXN-003": None,
        "TXN-005": 1000
    }

    if transaction_reference == "TXN-003":
        raise ExternalServiceError(
            "External settlement service unavailable."
        )

    return external_amounts.get(transaction_reference)


def reconcile_transactions(
    account_number,
    balance,
    bank_transactions
):
    reconciled_transactions = []
    mismatched_transactions = []
    failed_records = []

    for record in bank_transactions:

        try:
            # Validate the reconciliation record
            if not isinstance(record, dict):
                raise InvalidRecordError(
                    "Reconciliation record must be a dictionary."
                )

            reference = record.get("reference")
            bank_amount = record.get("bank_amount")

            if not reference:
                raise InvalidRecordError(
                    "Transaction reference is required."
                )

            if not isinstance(bank_amount, (int, float)):
                raise InvalidRecordError(
                    "Bank amount must be a number."
                )

            if bank_amount < 0:
                raise InvalidRecordError(
                    "Bank amount cannot be negative."
                )

            # Smallest risky statement:
            # external service call
            try:
                external_amount = get_external_amount(
                    reference
                )

            except ExternalServiceError as error:
                raise ReconciliationProcessingError(
                    "Unable to retrieve external settlement amount."
                ) from error

            # Compare the two amounts
            if bank_amount == external_amount:
                reconciled_transactions.append(
                    {
                        "reference": reference,
                        "amount": bank_amount
                    }
                )

            else:
                mismatched_transactions.append(
                    {
                        "reference": reference,
                        "bank_amount": bank_amount,
                        "external_amount": external_amount
                    }
                )

        except InvalidRecordError as error:

            failed_records.append(
                {
                    "reference": (
                        record.get("reference", "UNKNOWN")
                        if isinstance(record, dict)
                        else "UNKNOWN"
                    ),
                    "error": str(error),
                    "cause": None
                }
            )

        except ReconciliationProcessingError as error:

            failed_records.append(
                {
                    "reference": (
                        record.get("reference", "UNKNOWN")
                        if isinstance(record, dict)
                        else "UNKNOWN"
                    ),
                    "error": str(error),
                    "cause": (
                        str(error.__cause__)
                        if error.__cause__
                        else None
                    )
                }
            )

        # Continue processing the remaining records

    return {
        "account_number": account_number,
        "balance": balance,
        "reconciled": reconciled_transactions,
        "mismatched": mismatched_transactions,
        "failed": failed_records
    }


# Sample banking case
account_number = "ACC-1090"
balance = 10000

bank_transactions = [
    {"reference": "TXN-001", "bank_amount": 2000},
    {"reference": "TXN-002", "bank_amount": 3000},
    {"reference": "TXN-003", "bank_amount": 1500},
    {"reference": "TXN-004", "bank_amount": "abc"},
    {"reference": "TXN-005", "bank_amount": 1000}
]


try:
    result = reconcile_transactions(
        account_number,
        balance,
        bank_transactions
    )

    print("Account Number:", result["account_number"])
    print("Account Balance:", result["balance"])

    print("\nReconciled Transactions:")

    for transaction in result["reconciled"]:
        print(
            transaction["reference"],
            "-",
            transaction["amount"]
        )

    print("\nMismatched Transactions:")

    for transaction in result["mismatched"]:
        print(
            transaction["reference"],
            "- Bank:",
            transaction["bank_amount"],
            "| External:",
            transaction["external_amount"]
        )

    print("\nFailed Records:")

    for record in result["failed"]:
        print(
            record["reference"],
            "-",
            record["error"]
        )

        if record["cause"]:
            print(
                "  Original Cause:",
                record["cause"]
            )

except BankingError as error:
    print("Reconciliation failed:", error)

Account Number: ACC-1090
Account Balance: 10000

Reconciled Transactions:
TXN-001 - 2000
TXN-005 - 1000

Mismatched Transactions:
TXN-002 - Bank: 3000 | External: 2500

Failed Records:
TXN-003 - Unable to retrieve external settlement amount.
  Original Cause: External settlement service unavailable.
TXN-004 - Bank amount must be a number.


## === Q91. Design a banking exception hierarchy for banking case 3; use sample reference ACC-1091 and explain the result. ===

In [42]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class TransactionError(BankingError):
    """Base class for transaction-related errors."""
    pass


class InsufficientFundsError(TransactionError):
    """Raised when an account has insufficient funds."""
    pass


class InvalidTransactionError(TransactionError):
    """Raised when a transaction is invalid."""
    pass


class PaymentGatewayError(BankingError):
    """Raised when an external payment gateway fails."""
    pass


class TransactionProcessingError(BankingError):
    """Raised when transaction processing fails."""
    pass


def process_withdrawal(
    account_number,
    balance,
    withdrawal_amount
):
    try:
        # Business-rule validation
        if not isinstance(
            withdrawal_amount,
            (int, float)
        ):
            raise InvalidTransactionError(
                "Withdrawal amount must be a number."
            )

        if withdrawal_amount <= 0:
            raise InvalidTransactionError(
                "Withdrawal amount must be greater than zero."
            )

        if withdrawal_amount > balance:
            raise InsufficientFundsError(
                "Insufficient funds for this withdrawal."
            )

        # Update balance only after all validation succeeds
        balance -= withdrawal_amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "withdrawal_amount": withdrawal_amount,
            "remaining_balance": balance
        }

    except (
        InvalidTransactionError,
        InsufficientFundsError
    ):
        # Business-rule errors are already specific.
        raise

    except Exception as error:
        # Unexpected/programming/infrastructure error
        raise TransactionProcessingError(
            "Unexpected error while processing withdrawal."
        ) from error


# Sample banking case
account_number = "ACC-1091"
balance = 8000
withdrawal_amount = 10000


try:
    result = process_withdrawal(
        account_number,
        balance,
        withdrawal_amount
    )

    print("Withdrawal successful.")
    print("Account Number:", result["account_number"])
    print(
        "Withdrawal Amount:",
        result["withdrawal_amount"]
    )
    print(
        "Remaining Balance:",
        result["remaining_balance"]
    )

except InsufficientFundsError as error:
    print("Withdrawal rejected:", error)
    print("Account Number:", account_number)
    print("Account Balance:", balance)

except InvalidTransactionError as error:
    print("Invalid transaction:", error)

except TransactionProcessingError as error:
    print("Transaction processing failed:", error)
    print("Original Cause:", error.__cause__)

except BankingError as error:
    print("Banking error:", error)

Withdrawal rejected: Insufficient funds for this withdrawal.
Account Number: ACC-1091
Account Balance: 8000


## === Q92. Make a transfer rollback-safe for banking case 3; use sample reference ACC-1092 and explain the result. ===

In [43]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class TransferError(BankingError):
    """Base class for transfer-related errors."""
    pass


class InvalidTransferError(TransferError):
    """Raised when transfer details are invalid."""
    pass


class InsufficientFundsError(TransferError):
    """Raised when the source account has insufficient funds."""
    pass


class BeneficiaryUpdateError(TransferError):
    """Raised when the beneficiary update fails."""
    pass


class TransferProcessingError(BankingError):
    """Raised when an unexpected transfer error occurs."""
    pass


def update_beneficiary(beneficiary_balance, amount):
    """
    Simulate a beneficiary account update.

    The update fails to demonstrate rollback safety.
    """
    raise BeneficiaryUpdateError(
        "Beneficiary account update failed."
    )


def process_transfer(
    account_number,
    beneficiary_account,
    source_balance,
    beneficiary_balance,
    transfer_amount
):
    # Save original balances before making any changes
    original_source_balance = source_balance
    original_beneficiary_balance = beneficiary_balance

    try:
        # Validate transfer amount
        if not isinstance(
            transfer_amount,
            (int, float)
        ):
            raise InvalidTransferError(
                "Transfer amount must be a number."
            )

        if transfer_amount <= 0:
            raise InvalidTransferError(
                "Transfer amount must be greater than zero."
            )

        if transfer_amount > source_balance:
            raise InsufficientFundsError(
                "Insufficient funds in source account."
            )

        # Temporarily deduct from source account
        source_balance -= transfer_amount

        # Smallest risky statement
        try:
            beneficiary_balance = update_beneficiary(
                beneficiary_balance,
                transfer_amount
            )

        except BeneficiaryUpdateError as error:
            raise TransferProcessingError(
                "Transfer failed while updating beneficiary account."
            ) from error

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "beneficiary_account": beneficiary_account,
            "transfer_amount": transfer_amount,
            "source_balance": source_balance,
            "beneficiary_balance": beneficiary_balance
        }

    except (
        InvalidTransferError,
        InsufficientFundsError
    ):
        # Business-rule errors do not need wrapping
        raise

    except TransferProcessingError:
        # Roll back both balances
        source_balance = original_source_balance
        beneficiary_balance = original_beneficiary_balance

        raise

    except Exception as error:
        # Protect against unexpected failures
        source_balance = original_source_balance
        beneficiary_balance = original_beneficiary_balance

        raise TransferProcessingError(
            "Unexpected error while processing transfer."
        ) from error


# Sample banking case
account_number = "ACC-1092"
beneficiary_account = "BEN-1002"

source_balance = 10000
beneficiary_balance = 3000
transfer_amount = 4000

original_source_balance = source_balance
original_beneficiary_balance = beneficiary_balance


try:
    result = process_transfer(
        account_number,
        beneficiary_account,
        source_balance,
        beneficiary_balance,
        transfer_amount
    )

    print("Transfer successful.")
    print("Account Number:", result["account_number"])
    print(
        "Beneficiary Account:",
        result["beneficiary_account"]
    )
    print(
        "Transfer Amount:",
        result["transfer_amount"]
    )
    print(
        "Source Balance:",
        result["source_balance"]
    )
    print(
        "Beneficiary Balance:",
        result["beneficiary_balance"]
    )

except InvalidTransferError as error:
    print("Invalid transfer:", error)

except InsufficientFundsError as error:
    print("Transfer rejected:", error)

except TransferProcessingError as error:
    # Restore the balances in the calling scope
    source_balance = original_source_balance
    beneficiary_balance = original_beneficiary_balance

    print("Transfer failed:", error)
    print("Original Cause:", error.__cause__)

    print("\nRollback completed.")
    print("Account Number:", account_number)
    print("Source Balance:", source_balance)
    print(
        "Beneficiary Account:",
        beneficiary_account
    )
    print(
        "Beneficiary Balance:",
        beneficiary_balance
    )

Transfer failed: Transfer failed while updating beneficiary account.
Original Cause: Beneficiary account update failed.

Rollback completed.
Account Number: ACC-1092
Source Balance: 10000
Beneficiary Account: BEN-1002
Beneficiary Balance: 3000


## === Q93. Preserve causes across service layers for banking case 3; use sample reference ACC-1093 and explain the result. ===

In [44]:
class BankingError(Exception):
    """Base class for banking errors."""
    pass


class PaymentGatewayError(BankingError):
    """Raised when the payment gateway fails."""
    pass


class PaymentProcessingError(BankingError):
    """Raised when payment processing fails."""
    pass


def payment_gateway(payment_amount):
    """
    Simulate an external payment gateway.
    """

    if not isinstance(payment_amount, (int, float)):
        raise TypeError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise ValueError(
            "Payment amount must be greater than zero."
        )

    # Realistic infrastructure failure
    raise ConnectionError(
        "Payment gateway connection failed."
    )


def process_payment(
    account_number,
    balance,
    payment_amount
):
    try:
        # Smallest risky statement
        payment_gateway(payment_amount)

    except (TypeError, ValueError) as error:
        raise PaymentGatewayError(
            "Invalid payment information."
        ) from error

    except ConnectionError as error:
        raise PaymentGatewayError(
            "Payment gateway is unavailable."
        ) from error

    # Update balance only after successful gateway response
    balance -= payment_amount

    return {
        "status": "SUCCESS",
        "account_number": account_number,
        "payment_amount": payment_amount,
        "remaining_balance": balance
    }


def banking_service(
    account_number,
    balance,
    payment_amount
):
    try:
        return process_payment(
            account_number,
            balance,
            payment_amount
        )

    except PaymentGatewayError as error:
        raise PaymentProcessingError(
            "Payment processing failed."
        ) from error


# Sample banking case
account_number = "ACC-1093"
balance = 10000
payment_amount = 3000


try:
    result = banking_service(
        account_number,
        balance,
        payment_amount
    )

    print("Payment successful.")
    print("Account Number:", result["account_number"])
    print("Payment Amount:", result["payment_amount"])
    print(
        "Remaining Balance:",
        result["remaining_balance"]
    )

except PaymentProcessingError as error:
    print("Payment failed:", error)
    print("Immediate Cause:", error.__cause__)

    if error.__cause__:
        print(
            "Original Cause:",
            error.__cause__.__cause__
        )

    print("Account Number:", account_number)
    print("Account Balance:", balance)

except BankingError as error:
    print("Banking error:", error)

Payment failed: Payment processing failed.
Immediate Cause: Payment gateway is unavailable.
Original Cause: Payment gateway connection failed.
Account Number: ACC-1093
Account Balance: 10000


## === Q94. Implement bounded exponential retry for banking case 3; use sample reference ACC-1094 and explain the result. ===

In [45]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class InvalidPaymentError(BankingError):
    """Raised when payment details are invalid."""
    pass


class InsufficientFundsError(BankingError):
    """Raised when the account has insufficient funds."""
    pass


class TransientServiceError(BankingError):
    """Raised when an external service temporarily fails."""
    pass


class PaymentProcessingError(BankingError):
    """Raised when payment processing fails after all retries."""
    pass


def payment_service():
    """
    Simulate an external payment service.

    The service fails on the first 3 attempts
    and succeeds on the 4th attempt.
    """
    payment_service.attempts += 1

    if payment_service.attempts <= 3:
        raise TransientServiceError(
            "Payment service temporarily unavailable."
        )

    return True


payment_service.attempts = 0


def process_payment(
    account_number,
    balance,
    payment_amount,
    max_retries=4,
    base_delay=1,
    max_delay=4
):
    # Validate business rules before retrying.
    if not isinstance(payment_amount, (int, float)):
        raise InvalidPaymentError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise InvalidPaymentError(
            "Payment amount must be greater than zero."
        )

    if payment_amount > balance:
        raise InsufficientFundsError(
            "Insufficient funds."
        )

    last_error = None

    for attempt in range(1, max_retries + 1):

        try:
            # Smallest risky operation:
            # calling the external payment service.
            payment_service()

            # Update balance only after successful payment.
            balance -= payment_amount

            return {
                "status": "SUCCESS",
                "account_number": account_number,
                "payment_amount": payment_amount,
                "remaining_balance": balance,
                "attempts": attempt
            }

        except TransientServiceError as error:
            last_error = error

            delay = min(
                base_delay * (2 ** (attempt - 1)),
                max_delay
            )

            if attempt < max_retries:
                print(
                    f"Attempt {attempt} failed. "
                    f"Retrying after {delay} second(s)."
                )

            else:
                print(
                    f"Attempt {attempt} failed. "
                    f"Maximum retry limit reached."
                )

    # All retry attempts failed.
    raise PaymentProcessingError(
        "Payment failed after all retry attempts."
    ) from last_error


# Banking case 3
account_number = "ACC-1094"
balance = 10000
payment_amount = 3000

try:
    result = process_payment(
        account_number,
        balance,
        payment_amount
    )

    print("\nPayment successful.")
    print("Account Number:", result["account_number"])
    print("Payment Amount:", result["payment_amount"])
    print("Remaining Balance:", result["remaining_balance"])
    print("Attempts Used:", result["attempts"])

except InvalidPaymentError as error:
    print("\nInvalid payment:", error)

except InsufficientFundsError as error:
    print("\nPayment rejected:", error)

except PaymentProcessingError as error:
    print("\nPayment failed:", error)
    print("Original Cause:", error.__cause__)

except BankingError as error:
    print("\nBanking error:", error)

Attempt 1 failed. Retrying after 1 second(s).
Attempt 2 failed. Retrying after 2 second(s).
Attempt 3 failed. Retrying after 4 second(s).

Payment successful.
Account Number: ACC-1094
Payment Amount: 3000
Remaining Balance: 7000
Attempts Used: 4


## === Q95. Create a transaction context manager for banking case 3; use sample reference ACC-1095 and explain the result. ===

In [46]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class TransactionError(BankingError):
    """Base class for transaction-related errors."""
    pass


class InvalidAmountError(TransactionError):
    """Raised when an amount is invalid."""
    pass


class InsufficientFundsError(TransactionError):
    """Raised when there are insufficient funds."""
    pass


class TransactionProcessingError(BankingError):
    """Raised when an unexpected transaction error occurs."""
    pass


class TransactionContext:
    """Context manager for a rollback-safe banking transaction."""

    def __init__(self, balance):
        self.original_balance = balance
        self.balance = balance

    def __enter__(self):
        print("Transaction started.")
        return self

    def withdraw(self, amount):
        # Smallest risky operation:
        # validating and changing the transaction balance.
        if not isinstance(amount, (int, float)):
            raise InvalidAmountError(
                "Withdrawal amount must be a number."
            )

        if amount <= 0:
            raise InvalidAmountError(
                "Withdrawal amount must be greater than zero."
            )

        if amount > self.balance:
            raise InsufficientFundsError(
                "Insufficient funds."
            )

        self.balance -= amount

        print("Withdrawal completed:", amount)

    def deposit(self, amount):
        if not isinstance(amount, (int, float)):
            raise InvalidAmountError(
                "Deposit amount must be a number."
            )

        if amount <= 0:
            raise InvalidAmountError(
                "Deposit amount must be greater than zero."
            )

        self.balance += amount

        print("Deposit completed:", amount)

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            print("Transaction committed.")
            return False

        # Roll back all changes made inside the transaction.
        self.balance = self.original_balance

        print("Transaction rolled back.")
        print("Reason:", exc_value)

        # Preserve the original exception.
        return False


# Banking case 3
account_number = "ACC-1095"
starting_balance = 10000

try:
    transaction = TransactionContext(starting_balance)

    with transaction:
        transaction.withdraw(3000)
        transaction.deposit(1000)

    print("\nTransaction successful.")
    print("Account Number:", account_number)
    print("Original Balance:", starting_balance)
    print("Final Balance:", transaction.balance)

except InvalidAmountError as error:
    print("\nInvalid transaction:", error)

except InsufficientFundsError as error:
    print("\nTransaction rejected:", error)

except TransactionProcessingError as error:
    print("\nTransaction processing failed:", error)
    print("Original Cause:", error.__cause__)

except BankingError as error:
    print("\nBanking error:", error)

Transaction started.
Withdrawal completed: 3000
Deposit completed: 1000
Transaction committed.

Transaction successful.
Account Number: ACC-1095
Original Balance: 10000
Final Balance: 8000


## === Q96. Build a dead-letter queue for payment events for banking case 3; use sample reference ACC-1096 and explain the result. ===

In [47]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class InvalidPaymentError(BankingError):
    """Raised when a payment event is invalid."""
    pass


class TransientPaymentError(BankingError):
    """Raised when payment processing temporarily fails."""
    pass


class PaymentProcessingError(BankingError):
    """Raised when payment processing fails after retries."""
    pass


def payment_service(event):
    """
    Simulate an external payment service.

    TXN-003 fails on every attempt to demonstrate
    the dead-letter queue.
    """
    if event["reference"] == "TXN-003":
        raise TransientPaymentError(
            "Payment service temporarily unavailable."
        )

    return True


def process_payment_events(
    account_number,
    balance,
    payment_events,
    max_retries=2
):
    successful_payments = []
    dead_letter_queue = []

    for event in payment_events:

        try:
            # Validate the event.
            if not isinstance(event, dict):
                raise InvalidPaymentError(
                    "Payment event must be a dictionary."
                )

            reference = event.get("reference")
            amount = event.get("amount")

            if not reference:
                raise InvalidPaymentError(
                    "Payment reference is required."
                )

            if not isinstance(amount, (int, float)):
                raise InvalidPaymentError(
                    "Payment amount must be a number."
                )

            if amount <= 0:
                raise InvalidPaymentError(
                    "Payment amount must be greater than zero."
                )

            if amount > balance:
                raise InvalidPaymentError(
                    "Insufficient funds."
                )

            payment_succeeded = False
            last_error = None

            # Retry only temporary service failures.
            for attempt in range(1, max_retries + 1):

                try:
                    # Smallest risky operation:
                    # calling the external payment service.
                    payment_service(event)

                    payment_succeeded = True
                    break

                except TransientPaymentError as error:
                    last_error = error

                    print(
                        f"{reference}: "
                        f"Attempt {attempt} failed - {error}"
                    )

            # Move the event to the DLQ if all retries fail.
            if not payment_succeeded:
                raise PaymentProcessingError(
                    "Payment failed after all retry attempts."
                ) from last_error

            # Update balance only after successful processing.
            balance -= amount

            successful_payments.append(reference)

        except (
            InvalidPaymentError,
            PaymentProcessingError
        ) as error:

            dead_letter_queue.append(
                {
                    "reference": (
                        event.get("reference", "UNKNOWN")
                        if isinstance(event, dict)
                        else "UNKNOWN"
                    ),
                    "error": str(error),
                    "cause": (
                        str(error.__cause__)
                        if error.__cause__
                        else None
                    )
                }
            )

            # Continue with the next payment event.
            continue

    return (
        balance,
        successful_payments,
        dead_letter_queue
    )


# Banking case 3
account_number = "ACC-1096"
balance = 10000

payment_events = [
    {"reference": "TXN-001", "amount": 2000},
    {"reference": "TXN-002", "amount": "abc"},
    {"reference": "TXN-003", "amount": 3000},
    {"reference": "TXN-004", "amount": 4000},
    {"reference": "TXN-005", "amount": -500}
]


try:
    (
        final_balance,
        successful_payments,
        dead_letter_queue
    ) = process_payment_events(
        account_number,
        balance,
        payment_events
    )

    print("\nAccount Number:", account_number)
    print("Final Balance:", final_balance)

    print("\nSuccessful Payments:")

    for reference in successful_payments:
        print(reference)

    print("\nDead-Letter Queue:")

    for event in dead_letter_queue:
        print(
            event["reference"],
            "-",
            event["error"]
        )

        if event["cause"]:
            print(
                "  Original Cause:",
                event["cause"]
            )

except BankingError as error:
    print("Payment processing failed:", error)

TXN-003: Attempt 1 failed - Payment service temporarily unavailable.
TXN-003: Attempt 2 failed - Payment service temporarily unavailable.

Account Number: ACC-1096
Final Balance: 4000

Successful Payments:
TXN-001
TXN-004

Dead-Letter Queue:
TXN-002 - Payment amount must be a number.
TXN-003 - Payment failed after all retry attempts.
  Original Cause: Payment service temporarily unavailable.
TXN-005 - Payment amount must be greater than zero.


## === Q97. Make batch settlement failure-isolated for banking case 3; use sample reference ACC-1097 and explain the result. ===

In [48]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class InvalidSettlementError(BankingError):
    """Raised when settlement details are invalid."""
    pass


class SettlementServiceError(BankingError):
    """Raised when the external settlement service fails."""
    pass


class SettlementProcessingError(BankingError):
    """Raised when settlement processing fails."""
    pass


def settlement_service(transaction):
    """
    Simulate an external settlement service.

    TXN-003 fails to demonstrate failure isolation.
    """
    if transaction["reference"] == "TXN-003":
        raise SettlementServiceError(
            "Settlement service temporarily unavailable."
        )

    return True


def process_batch_settlement(
    account_number,
    balance,
    settlement_events
):
    successful_settlements = []
    failed_settlements = []

    for event in settlement_events:

        try:
            # Validate the settlement event.
            if not isinstance(event, dict):
                raise InvalidSettlementError(
                    "Settlement event must be a dictionary."
                )

            reference = event.get("reference")
            amount = event.get("amount")

            if not reference:
                raise InvalidSettlementError(
                    "Settlement reference is required."
                )

            if not isinstance(amount, (int, float)):
                raise InvalidSettlementError(
                    "Settlement amount must be a number."
                )

            if amount <= 0:
                raise InvalidSettlementError(
                    "Settlement amount must be greater than zero."
                )

            if amount > balance:
                raise InvalidSettlementError(
                    "Insufficient funds for settlement."
                )

            try:
                # Smallest risky operation:
                # calling the external settlement service.
                settlement_service(event)

            except SettlementServiceError as error:
                raise SettlementProcessingError(
                    "Settlement failed because the service was unavailable."
                ) from error

            # Update balance only after successful settlement.
            balance -= amount

            successful_settlements.append(
                {
                    "reference": reference,
                    "amount": amount
                }
            )

        except (
            InvalidSettlementError,
            SettlementProcessingError
        ) as error:

            failed_settlements.append(
                {
                    "reference": (
                        event.get("reference", "UNKNOWN")
                        if isinstance(event, dict)
                        else "UNKNOWN"
                    ),
                    "amount": (
                        event.get("amount", "UNKNOWN")
                        if isinstance(event, dict)
                        else "UNKNOWN"
                    ),
                    "error": str(error),
                    "cause": (
                        str(error.__cause__)
                        if error.__cause__
                        else None
                    )
                }
            )

            # Failure isolation:
            # continue with the next settlement.
            continue

    return (
        balance,
        successful_settlements,
        failed_settlements
    )


# Banking case 3
account_number = "ACC-1097"
balance = 10000

settlement_events = [
    {"reference": "TXN-001", "amount": 2000},
    {"reference": "TXN-002", "amount": "abc"},
    {"reference": "TXN-003", "amount": 3000},
    {"reference": "TXN-004", "amount": 4000},
    {"reference": "TXN-005", "amount": -500}
]


try:
    (
        final_balance,
        successful_settlements,
        failed_settlements
    ) = process_batch_settlement(
        account_number,
        balance,
        settlement_events
    )

    print("Account Number:", account_number)
    print("Final Balance:", final_balance)

    print("\nSuccessful Settlements:")

    for settlement in successful_settlements:
        print(
            settlement["reference"],
            "-",
            settlement["amount"]
        )

    print("\nFailed Settlements:")

    for settlement in failed_settlements:
        print(
            settlement["reference"],
            "-",
            settlement["error"]
        )

        if settlement["cause"]:
            print(
                "  Original Cause:",
                settlement["cause"]
            )

except BankingError as error:
    print("Batch settlement failed:", error)

Account Number: ACC-1097
Final Balance: 4000

Successful Settlements:
TXN-001 - 2000
TXN-004 - 4000

Failed Settlements:
TXN-002 - Settlement amount must be a number.
TXN-003 - Settlement failed because the service was unavailable.
  Original Cause: Settlement service temporarily unavailable.
TXN-005 - Settlement amount must be greater than zero.


## === Q98. Report exact paths in nested statements for banking case 3; use sample reference ACC-1098 and explain the result. ===

In [49]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class PaymentValidationError(BankingError):
    """Raised when payment validation fails."""
    pass


class TransactionProcessingError(BankingError):
    """Raised when transaction processing fails."""
    pass


def validate_payment(payment_amount):
    """Validate the payment amount."""

    if not isinstance(payment_amount, (int, float)):
        raise TypeError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise ValueError(
            "Payment amount must be greater than zero."
        )

    return True


def process_payment(
    account_number,
    balance,
    payment_amount
):
    try:

        # Inner layer: validate the payment.
        try:
            validate_payment(payment_amount)

        except (TypeError, ValueError) as error:
            raise PaymentValidationError(
                "Payment validation failed."
            ) from error

        # Balance is updated only after validation succeeds.
        balance -= payment_amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "payment_amount": payment_amount,
            "remaining_balance": balance
        }

    except PaymentValidationError as error:
        # Outer layer adds transaction context.
        raise TransactionProcessingError(
            "Transaction processing failed."
        ) from error


# Banking case 3
account_number = "ACC-1098"
balance = 10000
payment_amount = "abc"


try:
    result = process_payment(
        account_number,
        balance,
        payment_amount
    )

    print("Payment successful.")
    print("Account Number:", result["account_number"])
    print("Payment Amount:", result["payment_amount"])
    print(
        "Remaining Balance:",
        result["remaining_balance"]
    )

except TransactionProcessingError as error:
    print("Transaction failed:", error)

    # First level of the exception chain.
    print("Immediate Cause:", error.__cause__)

    # Second level of the exception chain.
    if error.__cause__:
        print(
            "Original Cause:",
            error.__cause__.__cause__
        )

    print("Account Number:", account_number)
    print("Account Balance:", balance)

except BankingError as error:
    print("Banking error:", error)

Transaction failed: Transaction processing failed.
Immediate Cause: Payment validation failed.
Original Cause: Payment amount must be a number.
Account Number: ACC-1098
Account Balance: 10000


## === Q99. Design idempotent recovery after timeout for banking case 3; use sample reference ACC-1099 and explain the result. ===

In [50]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class InvalidPaymentError(BankingError):
    """Raised when payment details are invalid."""
    pass


class PaymentTimeoutError(BankingError):
    """Raised when the payment response times out."""
    pass


class PaymentProcessingError(BankingError):
    """Raised when payment processing fails."""
    pass


# Simulated record of payments already processed
processed_transactions = set()


def payment_service(transaction_reference, amount):
    """
    Simulate an external payment service.

    The first request for TXN-001 is actually completed,
    but the response times out.
    """

    # If the transaction was already processed,
    # return the existing result.
    if transaction_reference in processed_transactions:
        return "ALREADY_PROCESSED"

    # Simulate the service completing the payment
    # before the response reaches our application.
    processed_transactions.add(transaction_reference)

    raise PaymentTimeoutError(
        "Payment completed, but the response timed out."
    )


def process_payment(
    account_number,
    balance,
    transaction_reference,
    payment_amount
):
    # Validate business rules first.
    if not transaction_reference:
        raise InvalidPaymentError(
            "Transaction reference is required."
        )

    if not isinstance(payment_amount, (int, float)):
        raise InvalidPaymentError(
            "Payment amount must be a number."
        )

    if payment_amount <= 0:
        raise InvalidPaymentError(
            "Payment amount must be greater than zero."
        )

    if payment_amount > balance:
        raise InvalidPaymentError(
            "Insufficient funds."
        )

    try:
        # Smallest risky statement:
        # calling the external payment service.
        payment_service(
            transaction_reference,
            payment_amount
        )

        # This point is reached only if the service
        # completes normally.
        balance -= payment_amount

        return {
            "status": "SUCCESS",
            "account_number": account_number,
            "transaction_reference": transaction_reference,
            "remaining_balance": balance
        }

    except PaymentTimeoutError as error:

        print("Timeout detected.")
        print("Original Cause:", error)

        # Recovery check using the unique transaction reference.
        if transaction_reference in processed_transactions:

            print(
                "Payment was already processed by the service."
            )

            # Apply the balance update exactly once
            # during recovery.
            balance -= payment_amount

            return {
                "status": "RECOVERED",
                "account_number": account_number,
                "transaction_reference": transaction_reference,
                "remaining_balance": balance
            }

        raise PaymentProcessingError(
            "Unable to determine payment status."
        ) from error


# Banking case 3
account_number = "ACC-1099"
balance = 10000
transaction_reference = "TXN-001"
payment_amount = 3000


try:
    result = process_payment(
        account_number,
        balance,
        transaction_reference,
        payment_amount
    )

    print("\nPayment Result:")
    print("Status:", result["status"])
    print("Account Number:", result["account_number"])
    print(
        "Transaction Reference:",
        result["transaction_reference"]
    )
    print(
        "Remaining Balance:",
        result["remaining_balance"]
    )

except InvalidPaymentError as error:
    print("\nInvalid payment:", error)

except PaymentProcessingError as error:
    print("\nPayment failed:", error)
    print("Original Cause:", error.__cause__)

except BankingError as error:
    print("\nBanking error:", error)

Timeout detected.
Original Cause: Payment completed, but the response timed out.
Payment was already processed by the service.

Payment Result:
Status: RECOVERED
Account Number: ACC-1099
Transaction Reference: TXN-001
Remaining Balance: 7000


## === Q100. Build an exception-safe reconciliation pipeline for banking case 3; use sample reference ACC-1100 and explain the result. ===

In [51]:
class BankingError(Exception):
    """Base class for all banking errors."""
    pass


class ReconciliationError(BankingError):
    """Base class for reconciliation errors."""
    pass


class InvalidRecordError(ReconciliationError):
    """Raised when a reconciliation record is invalid."""
    pass


class ExternalServiceError(ReconciliationError):
    """Raised when the external settlement service fails."""
    pass


class ReconciliationProcessingError(ReconciliationError):
    """Raised when reconciliation processing fails."""
    pass


def get_external_amount(transaction_reference):
    """
    Simulate an external settlement service.

    TXN-003 fails to demonstrate exception-safe
    reconciliation.
    """

    external_amounts = {
        "TXN-001": 2000,
        "TXN-002": 2500,
        "TXN-003": None,
        "TXN-005": 1000
    }

    # Smallest risky operation:
    # retrieving data from the external service.
    if transaction_reference == "TXN-003":
        raise ExternalServiceError(
            "External settlement service unavailable."
        )

    return external_amounts.get(transaction_reference)


def reconcile_transactions(
    account_number,
    balance,
    bank_transactions
):
    reconciled_transactions = []
    mismatched_transactions = []
    failed_records = []

    for record in bank_transactions:

        try:
            # Validate the record.
            if not isinstance(record, dict):
                raise InvalidRecordError(
                    "Reconciliation record must be a dictionary."
                )

            reference = record.get("reference")
            bank_amount = record.get("bank_amount")

            if not reference:
                raise InvalidRecordError(
                    "Transaction reference is required."
                )

            if not isinstance(bank_amount, (int, float)):
                raise InvalidRecordError(
                    "Bank amount must be a number."
                )

            if bank_amount < 0:
                raise InvalidRecordError(
                    "Bank amount cannot be negative."
                )

            try:
                external_amount = get_external_amount(
                    reference
                )

            except ExternalServiceError as error:
                raise ReconciliationProcessingError(
                    "Unable to retrieve external settlement amount."
                ) from error

            # Compare bank and external amounts.
            if bank_amount == external_amount:
                reconciled_transactions.append(
                    {
                        "reference": reference,
                        "amount": bank_amount
                    }
                )

            else:
                mismatched_transactions.append(
                    {
                        "reference": reference,
                        "bank_amount": bank_amount,
                        "external_amount": external_amount
                    }
                )

        except InvalidRecordError as error:

            failed_records.append(
                {
                    "reference": (
                        record.get("reference", "UNKNOWN")
                        if isinstance(record, dict)
                        else "UNKNOWN"
                    ),
                    "error": str(error),
                    "cause": None
                }
            )

            # Continue processing the remaining records.
            continue

        except ReconciliationProcessingError as error:

            failed_records.append(
                {
                    "reference": (
                        record.get("reference", "UNKNOWN")
                        if isinstance(record, dict)
                        else "UNKNOWN"
                    ),
                    "error": str(error),
                    "cause": (
                        str(error.__cause__)
                        if error.__cause__
                        else None
                    )
                }
            )

            # Continue processing the remaining records.
            continue

    return {
        "account_number": account_number,
        "balance": balance,
        "reconciled": reconciled_transactions,
        "mismatched": mismatched_transactions,
        "failed": failed_records
    }


# Banking case 3
account_number = "ACC-1100"
balance = 10000

bank_transactions = [
    {"reference": "TXN-001", "bank_amount": 2000},
    {"reference": "TXN-002", "bank_amount": 3000},
    {"reference": "TXN-003", "bank_amount": 1500},
    {"reference": "TXN-004", "bank_amount": "abc"},
    {"reference": "TXN-005", "bank_amount": 1000}
]


try:
    result = reconcile_transactions(
        account_number,
        balance,
        bank_transactions
    )

    print("Account Number:", result["account_number"])
    print("Account Balance:", result["balance"])

    print("\nReconciled Transactions:")

    for transaction in result["reconciled"]:
        print(
            transaction["reference"],
            "-",
            transaction["amount"]
        )

    print("\nMismatched Transactions:")

    for transaction in result["mismatched"]:
        print(
            transaction["reference"],
            "- Bank:",
            transaction["bank_amount"],
            "| External:",
            transaction["external_amount"]
        )

    print("\nFailed Records:")

    for record in result["failed"]:
        print(
            record["reference"],
            "-",
            record["error"]
        )

        if record["cause"]:
            print(
                "  Original Cause:",
                record["cause"]
            )

except BankingError as error:
    print("Reconciliation failed:", error)

Account Number: ACC-1100
Account Balance: 10000

Reconciled Transactions:
TXN-001 - 2000
TXN-005 - 1000

Mismatched Transactions:
TXN-002 - Bank: 3000 | External: 2500

Failed Records:
TXN-003 - Unable to retrieve external settlement amount.
  Original Cause: External settlement service unavailable.
TXN-004 - Bank amount must be a number.
